# 신용카드 이상거래 탐지 — 팀 통합 노트북

다섯 명이 각자 진행한 노트북을 하나로 합친 것이다. 데이터는 맨 앞 **공통 데이터 로드** 셀에서 한 번만 읽어 `_raw`에 두고, 각 담당 섹션은 `train_df = _raw.copy()`로 시작한다. 따라서 한 사람의 전처리(중복 제거·Time 제거·스케일링 등)가 다음 사람 섹션에 영향을 주지 않는다.

| 담당 | 모델 | 원본 노트북 |
|---|---|---|
| 전유경 | CatBoost | `reports/YG/JYG_CatBoost.ipynb` |
| 이혜현 | LightGBM | `reports/HH/LHH_02_credit_ldbm.ipynb` |
| 이진희 | XGBoost | `reports/JH/LJH check copy.ipynb`, `LJH check kfold추가.ipynb` |
| 김소현 | Logistic Regression | `reports/SH/SH_CreditCard.ipynb` |
| 권용우 | Logistic Regression | `reports/YW/YWK(1).ipynb` |

**실행 순서**: 아래 *공통 데이터 로드* 셀을 먼저 실행한 뒤, 원하는 담당 섹션을 위에서부터 실행한다. 각 섹션은 독립적이라 순서를 바꿔도 된다(단, 이진희의 *5-Fold* 는 그 앞 XGBoost 섹션과 같은 `X_features`/`y_labels` 정의를 다시 만들어 자체 완결).

> 원본 노트북들의 변수명이 제각각(`train_df`·`df`·`card_df`)이라 통합본에서는 모두 `train_df`로 통일했다. 데이터 로드 경로는 이 노트북 위치(`02_classification_credit_card/`) 기준 `../data/creditcard.csv`.

## 공통 데이터 로드

In [ ]:
# ── 공통: 라이브러리 + 원본 데이터 1회 로드 ──
# 각 담당 섹션 실행 전에 이 셀을 먼저 돌린다. 이후 섹션들은 여기서 만든 _raw 를 copy() 해서 쓴다.
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'   # 그래프 한글 깨짐 방지 (Windows 맑은 고딕)
plt.rcParams['axes.unicode_minus'] = False

_raw = pd.read_csv('../data/creditcard.csv')     # 31개 열: Time, V1~V28, Amount, Class
print('원본 데이터:', _raw.shape)
_raw.head(3)

## [전유경] CatBoost

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# matplotlib 기본 폰트는 한글을 지원하지 않아 그래프의 한글 라벨/제목이 깨져 보인다.
# Windows에 기본 설치된 한글 폰트(맑은 고딕)로 지정하고, 마이너스 기호 깨짐도 함께 방지한다.
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:

print(f"Train 데이터 크기: {train_df.shape}")

train_df.head()

In [ ]:
train_df.describe()

In [ ]:
# Class 불균형 정도를 한눈에 확인하기 위해 막대그래프로 시각화한다.
# 사기(Class=1) 비율이 매우 작아 이후 전처리·모델링에서 불균형을 고려해야 함을 보여준다.
class_counts = train_df['Class'].value_counts().sort_index()
class_pct = train_df['Class'].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['정상(0)', '사기(1)'], class_counts.values, color=['#4C72B0', '#C44E52'])
for bar, pct in zip(bars, class_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{bar.get_height():,.0f}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('거래 건수')
ax.set_title('Class 분포 (정상 vs 사기)')
# 막대 위 수치 라벨이 제목과 겹치지 않도록 y축 상단 여백을 넉넉히 확보한다.
ax.set_ylim(0, class_counts.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# 완전히 동일한 값을 가진 중복 행이 존재하면, train/test 분할 시 같은 거래가
# 양쪽에 나뉘어 들어가 모델이 테스트셋의 데이터를 학습 과정에서 이미 보게 되는
# 데이터 누수(data leakage)가 발생해 성능이 실제보다 부풀려질 수 있다.
# 따라서 다른 전처리보다 먼저 중복 행을 제거한다.
n_before = len(train_df)
dup_count = train_df.duplicated().sum()
print(f"중복 행 개수: {dup_count}")

train_df.drop_duplicates(inplace=True)
print(f"중복 제거 후 데이터 크기: {train_df.shape}")

In [ ]:
# 중복 제거로 데이터 크기가 얼마나 줄었는지 막대그래프로 비교한다.
fig, ax = plt.subplots(figsize=(5, 4))
sizes = [n_before, len(train_df)]
bars = ax.bar(['제거 전', '제거 후'], sizes, color=['#8C8C8C', '#55A868'])
for bar, v in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f'{v:,}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('행 수')
ax.set_title(f'중복 제거 전후 데이터 크기 비교 (중복 {dup_count}건 제거)')
plt.tight_layout()
plt.show()

In [ ]:
# 전처리 전후 비교 시각화를 위해 원본 데이터를 별도로 보존해 둔다.
card_df_before = train_df.copy()

# V14는 Class(사기 여부)와 상관관계가 가장 높은 피처 중 하나이고,
# 사기(Class=1) 거래만 놓고 봤을 때 분포가 정규분포에 가장 가까워
# IQR(사분위범위) 기반 이상치 탐지에 적합한 피처이다.
v14_fraud = train_df['V14'].loc[train_df['Class'] == 1].values
q25, q75 = np.percentile(v14_fraud, 25), np.percentile(v14_fraud, 75)
iqr = q75 - q25

# 이상치 기준: Q1 - 1.5*IQR ~ Q3 + 1.5*IQR 범위를 벗어나는 값
cutoff = iqr * 1.5
lower, upper = q25 - cutoff, q75 + cutoff
print(f"V14 이상치 하한: {lower:.4f}, 상한: {upper:.4f}")

# 정상 거래(Class=0)의 V14 극단값은 정상적인 변동일 수 있으므로,
# 사기(Class=1) 거래 중에서 임계값을 벗어난 값만 이상치로 판단해 제거한다.
outlier_index = train_df[(train_df['Class'] == 1) & ((train_df['V14'] < lower) | (train_df['V14'] > upper))].index
print(f"V14 이상치 개수: {len(outlier_index)}")

train_df = train_df.drop(outlier_index)
print(f"이상치 제거 후 데이터 크기: {train_df.shape}")

In [ ]:
# Time은 첫 거래로부터 경과된 초(seconds)를 나타내는 값으로,
# 사기 여부와 직접적인 인과관계가 없는 단순 순번/타임스탬프성 피처이다.
# 이 값을 그대로 학습에 사용하면 모델이 특정 시간대에 과적합될 위험이 있어 제거한다.
train_df.drop('Time', axis=1, inplace=True)
print(f"Time 컬럼 제거 후 데이터 크기: {train_df.shape}")

In [ ]:
# 전처리 전(원본) vs 전처리 후(V14 이상치 4건 제거) V14 분포를 boxplot으로 비교한다.
# Class=1(사기) 쪽 하단의 극단적인 이상치가 제거된 것을 시각적으로 확인한다.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x='Class', y='V14', data=card_df_before, ax=axes[0])
axes[0].set_title('전처리 전: V14 vs Class')

sns.boxplot(x='Class', y='V14', data=train_df, ax=axes[1])
axes[1].set_title('전처리 후: V14 vs Class (이상치 4건 제거)')

plt.tight_layout()
plt.show()

In [ ]:
# Class=1(사기) 비율이 0.17% 수준인 극심한 불균형 데이터이므로,
# train/test로 나눌 때 stratify=y로 두 세트의 클래스 비율을 동일하게 유지한다.
X = train_df.drop('Class', axis=1)
y = train_df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"Train Class 비율:\n{y_train.value_counts(normalize=True)}")
print(f"Test Class 비율:\n{y_test.value_counts(normalize=True)}")

In [ ]:
# CatBoost는 트리 기반 부스팅 모델로, 별도의 스케일링 없이도 안정적으로 학습되고
# 과적합 방지(oblivious tree, ordered boosting)에 강점이 있어 사용한다.
# eval_metric은 학습 중 eval_set 기준으로 가장 좋은 iteration(스냅샷)을 선택하는 기준이 되는데,
# AUC 기준으로 고른 모델보다 F1 기준으로 고른 모델이 precision 손실 없이(FP 동일) 놓친 사기(FN)를
# 1건 더 줄여주는 것을 실험으로 확인해 eval_metric='F1'을 사용한다.
cat_model = CatBoostClassifier(
    random_state=0,
    eval_metric='F1',
    verbose=100
)
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

pred = cat_model.predict(X_test)
pred_proba = cat_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

# 클래스별 세부 지표는 classification_report에 담겨 있지만,
# 불균형 데이터에서 특히 중요한 지표(사기 클래스 기준 정확도/정밀도/재현율/F1/AUC)는
# 한눈에 비교할 수 있도록 별도로도 출력한다.
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
roc_score = roc_auc_score(y_test, pred_proba)

print("=" * 40)
print(f"CatBoost 테스트 세트 정확도(Accuracy): {accuracy:.4f}")
print(f"CatBoost 테스트 세트 정밀도(Precision): {precision:.4f}")
print(f"CatBoost 테스트 세트 재현율(Recall): {recall:.4f}")
print(f"CatBoost 테스트 세트 F1-score: {f1:.4f}")
print(f"CatBoost 테스트 세트 ROC-AUC: {roc_score:.4f}")
print("=" * 40)

In [ ]:
# confusion_matrix를 숫자 배열로만 보면 오탐/미탐 위치를 직관적으로 파악하기 어려워
# heatmap으로 시각화한다.
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['정상(0)', '사기(1)'], yticklabels=['정상(0)', '사기(1)'], ax=ax)
ax.set_xlabel('예측값')
ax.set_ylabel('실제값')
ax.set_title('CatBoost Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve와 Precision-Recall Curve를 함께 그려 임계값 전반에 걸친 성능을 확인한다.
# 특히 Precision-Recall Curve는 사기 비율이 0.16% 수준인 불균형 데이터에서
# ROC-AUC보다 클래스 불균형의 영향을 더 잘 드러내는 지표다.
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_test, pred_proba)
precision_arr, recall_arr, _ = precision_recall_curve(y_test, pred_proba)
ap_score = average_precision_score(y_test, pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color='#C44E52', label=f'ROC curve (AUC = {roc_score:.4f})')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')

axes[1].plot(recall_arr, precision_arr, color='#4C72B0', label=f'PR curve (AP = {ap_score:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
# CatBoost가 어떤 피처를 기준으로 사기를 판단했는지 확인하기 위해 피처 중요도를 시각화한다.
feat_imp = pd.Series(cat_model.get_feature_importance(), index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(x=feat_imp.head(15).values, y=feat_imp.head(15).index, color='#4C72B0', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('CatBoost Feature Importance (Top 15)')
plt.tight_layout()
plt.show()

In [ ]:
# 정확도/정밀도/재현율/F1/AUC를 한 그래프에 모아 최종 성능을 한눈에 비교한다.
metrics_summary = pd.Series({
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1,
    'ROC-AUC': roc_score,
})

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(metrics_summary.index, metrics_summary.values, color='#55A868')
for bar, v in zip(bars, metrics_summary.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f'{v:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_title('CatBoost 최종 성능 지표 요약')
plt.tight_layout()
plt.show()

## [이혜현] LightGBM

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

train_df.head(3)

In [ ]:
# 상위 5개 행 미리보기
train_df.head()

In [ ]:
# 결측치 및 데이터 타입, 메모리 사용량 확인
train_df.info()

In [ ]:
# 기술통계량 요약 (평균, 표준편차, 사분위수 등)
train_df.describe()

In [ ]:
# 클래스별(0과 1) 건수 및 비율 확인
counts = train_df['Class'].value_counts()
percents = train_df['Class'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'Count': counts, 'Percentage(%)': percents})
summary

In [ ]:
# 정상(0)과 사기(1) 거래별로 기술통계량 분리해서 확인
print("=== [정상 거래 Class 0] V1 ~ V5 통계량 ===")
print(train_df[train_df['Class'] == 0][['V8']].describe())

print("\n=== [사기 거래 Class 1] V1 ~ V5 통계량 ===")
print(train_df[train_df['Class'] == 1][['V8']].describe())

In [ ]:
# 전체 결측치 개수
train_df.isna().sum().sum() 

In [ ]:
# 중복치 제거 (1,081건 정리)
initial_count = len(train_df)
train_df = train_df.drop_duplicates().reset_index(drop=True)
print(f"중복치 제거 완료: {initial_count - len(train_df)}건 삭제 (남은 행: {len(train_df):,}개)")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

# 1. IQR 기반 이상치 인덱스 반환 함수
def get_outlier(df=None, column=None, weight=1.5):
    # Class=1 (사기 거래) 데이터만 추출
    fraud = df[df['Class'] == 1][column]
    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)
    
    iqr = quantile_75 - quantile_25
    iqr_weight = iqr * weight
    lowest_val = quantile_25 - iqr_weight
    highest_val = quantile_75 + iqr_weight
    
    # 사분위수 범위를 벗어난 인덱스 추출
    outlier_index = fraud[(fraud < lowest_val) | (fraud > highest_val)].index
    return outlier_index

In [ ]:
# 2. 종합 전처리 함수 (Amount 스케일링 + Time/Amount 드랍 + V14 이상치 제거)
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    
    # Amount RobustScaler 적용 후 맨 앞 컬럼에 삽입
    rob_scaler = RobustScaler()
    amount_n = rob_scaler.fit_transform(df_copy['Amount'].values.reshape(-1, 1))
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    
    # 기존 Time, Amount 컬럼 삭제
    cols_to_drop = [col for col in ['Time', 'Amount'] if col in df_copy.columns]
    df_copy.drop(columns=cols_to_drop, inplace=True)
    
    # V14 이상치 행 삭제
    outlier_index = get_outlier(df=df_copy, column='V14', weight=1.5)
    print(f"제거된 V14 이상치 개수: {len(outlier_index)}건")
    df_copy.drop(outlier_index, axis=0, inplace=True)
    
    return df_copy

In [ ]:
# 전처리 함수 실행
train_df_processed = get_preprocessed_df(train_df)

print(f"전처리 후 데이터 크기: {train_df_processed.shape}")
print(f"전처리 후 컬럼 목록:\n{list(train_df_processed.columns)}")

In [ ]:
from sklearn.model_selection import train_test_split

# 1. 피처(X_features)와 타깃(y_labels) 분리
# Class를 제외한 컬럼들이 피처
X_features = train_df_processed.drop(columns=['Class'])
y_labels = train_df_processed['Class']

# 2. 8:2 층화 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    y_labels, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_labels
)

print(f"학습용 피처(X_train) 크기: {X_train.shape}")
print(f"검증용 피처(X_test) 크기: {X_test.shape}")
print(f"Train 사기 거래 비율: {y_train.mean():.4%}")
print(f"Test 사기 거래 비율: {y_test.mean():.4%}")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

def get_clf_eval(y_test, y_pred, y_pred_proba):
    roc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    
    print(f"ROC-AUC  : {roc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    print("\n[혼동 행렬 (Confusion Matrix)]")
    print(confusion_matrix(y_test, y_pred))
    print("\n[분류 리포트]")
    print(classification_report(y_test, y_pred, digits=4))

In [ ]:
from lightgbm import LGBMClassifier

# 1. LightGBM 모델 생성 (불균형 데이터 필수 옵션 설정)
lgbm_clf = LGBMClassifier(
    n_estimators=1000, 
    num_leaves=64, 
    n_jobs=-1, 
    boost_from_average=False, 
    random_state=42,
    verbose=-1
)

# 2. 학습 진행
lgbm_clf.fit(X_train, y_train)

# 3. 예측 (이진 분류 결과 및 확률값)
y_pred = lgbm_clf.predict(X_test)
y_pred_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 4. 성능 지표 출력
print("=== [LightGBM 전처리 후 평가 결과] ===")
get_clf_eval(y_test, y_pred, y_pred_proba)

In [ ]:
from sklearn.preprocessing import Binarizer

# 임곗값 후보군 설정 (0.1부터 0.9까지 0.1 간격)
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("=== [임곗값별 성능 변화] ===")
for custom_threshold in thresholds:
    # 확률값을 기준으로 새로운 임곗값 적용
    custom_predict = (y_pred_proba >= custom_threshold).astype(int)
    
    # 혼동 행렬에서 TP, FP, FN 추출
    cm = confusion_matrix(y_test, custom_predict)
    precision = cm[1, 1] / (cm[0, 1] + cm[1, 1]) if (cm[0, 1] + cm[1, 1]) > 0 else 0
    recall = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"임곗값: {custom_threshold:.1f} | 정밀도(Precision): {precision:.4f} | 재현율(Recall): {recall:.4f} (사기 {cm[1,1]}/{cm[1,0]+cm[1,1]}건 검출) | F1: {f1:.4f} | 오탐(FP): {cm[0,1]}건")

In [ ]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

# 1. 탐색 공간(Search Space) 설정
space = {
    'num_leaves': hp.choice('num_leaves', [31, 64, 128]),
    'max_depth': hp.choice('max_depth', [5, 7, 10, -1]),
    'min_child_samples': hp.choice('min_child_samples', [10, 20, 30, 50]),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0)
}

# 2. 목적 함수 (Objective Function) 정의
def objective(params):
    model = LGBMClassifier(
        **params,
        n_estimators=400,
        boost_from_average=False,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    
    # 3-Fold 층화 교차 검증
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    proba_cv = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    
    # PR-AUC 평가
    score = average_precision_score(y_train, proba_cv)
    
    # fmin은 최소화하므로 음수로 반환
    return {'loss': -score, 'status': STATUS_OK}

# 3. Hyperopt 최적화 실행 (50회 탐색)
trials = Trials()
best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Hyperopt 탐색 완료!")
print("최적 인덱스/값:", best_params)

In [ ]:
# hp.choice 인덱스 매핑 복원
num_leaves_list = [31, 64, 128]
max_depth_list = [5, 7, 10, -1]
min_child_samples_list = [10, 20, 30, 50]

best_lgbm_params = {
    'num_leaves': num_leaves_list[best_params['num_leaves']],
    'max_depth': max_depth_list[best_params['max_depth']],
    'min_child_samples': min_child_samples_list[best_params['min_child_samples']],
    'learning_rate': best_params['learning_rate'],
    'subsample': best_params['subsample'],
    'colsample_bytree': best_params['colsample_bytree'],
    'n_estimators': 1000,
    'boost_from_average': False,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

# 최적 모델 생성 및 학습
tuned_lgbm = LGBMClassifier(**best_lgbm_params)
tuned_lgbm.fit(X_train, y_train)

# Test 데이터 예측
y_pred_tuned = tuned_lgbm.predict(X_test)
y_pred_proba_tuned = tuned_lgbm.predict_proba(X_test)[:, 1]

# 최종 결과 출력
print("=== [LightGBM Hyperopt 튜닝 후 최종 결과] ===")
get_clf_eval(y_test, y_pred_tuned, y_pred_proba_tuned)

## LightGBM 모델 개선 실험

다양한 방법으로 LightGBM 모델의 성능을 개선하고 비교합니다.

### 실험 계획

1. **Baseline**: 현재 기본 LightGBM
2. **클래스 가중치**: scale_pos_weight로 불균형 데이터 처리
3. **균형 샘플 앙상블**: Fraud:Normal = 1:10 비율로 여러 샘플 생성 후 앙상블
4. **K-Fold 앙상블**: 5-Fold 교차검증으로 모델 앙상블
5. **Early Stopping**: Validation set 기반 조기 종료
6. **임곗값 최적화**: F1-Score 기준 최적 임곗값 찾기

In [ ]:
# 실험 결과 저장소
experiment_results = {}

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import copy

print("\n" + "="*70)
print("LightGBM 모델 개선 실험 시작")
print("="*70)

### Experiment 1: Baseline (현재 모델)

In [ ]:
print("\n[Exp 1] Baseline 모델")
print("-" * 70)

# 기존 모델 사용 (위에서 학습된 lgbm_clf)
y_pred_exp1 = lgbm_clf.predict(X_test)
y_pred_proba_exp1 = lgbm_clf.predict_proba(X_test)[:, 1]

roc_exp1 = roc_auc_score(y_test, y_pred_proba_exp1)
pr_exp1 = average_precision_score(y_test, y_pred_proba_exp1)
f1_exp1 = f1_score(y_test, y_pred_exp1)

print(f"설정: 기본 LightGBM (num_leaves=64, n_estimators=1000)")
print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp1:.4f}")
print(f"  PR-AUC:  {pr_exp1:.4f}")
print(f"  F1-Score: {f1_exp1:.4f}")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp1))

experiment_results['baseline'] = {
    'name': 'Baseline (현재 모델)',
    'description': '기본 LightGBM 설정',
    'roc_auc': roc_exp1,
    'pr_auc': pr_exp1,
    'f1_score': f1_exp1,
    'params': 'num_leaves=64, n_estimators=1000'
}

### Experiment 2: 클래스 가중치 조정 (scale_pos_weight)

**왜 이 방법을 썼나?**
- 데이터의 클래스 불균형이 심함 (정상:사기 = 99.8:0.2)
- scale_pos_weight를 조정하면 소수 클래스(사기)에 더 많은 가중치 부여
- 결과: 사기 탐지율 향상

**예상 효과:**
- PR-AUC 향상 가능
- 사기 탐지 재현율 증가

In [ ]:
print("\n[Exp 2] 클래스 가중치 조정")
print("-" * 70)

# 클래스 비율 계산
fraud_count = (y_train == 1).sum()
normal_count = (y_train == 0).sum()
scale_pos_weight = normal_count / fraud_count

print(f"클래스 비율: 정상={normal_count:,}개, 사기={fraud_count:,}개")
print(f"scale_pos_weight = {scale_pos_weight:.2f}")
print(f"\n의미: 사기(양성) 클래스에 {scale_pos_weight:.2f}배 가중치 부여")

model_exp2 = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    boost_from_average=False,
    random_state=42,
    verbose=-1
)

model_exp2.fit(X_train, y_train)
y_pred_exp2 = model_exp2.predict(X_test)
y_pred_proba_exp2 = model_exp2.predict_proba(X_test)[:, 1]

roc_exp2 = roc_auc_score(y_test, y_pred_proba_exp2)
pr_exp2 = average_precision_score(y_test, y_pred_proba_exp2)
f1_exp2 = f1_score(y_test, y_pred_exp2)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp2:.4f} (Baseline 대비: {roc_exp2-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp2:.4f} (Baseline 대비: {pr_exp2-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp2:.4f} (Baseline 대비: {f1_exp2-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp2))

experiment_results['class_weight'] = {
    'name': '클래스 가중치 조정',
    'description': f'scale_pos_weight={scale_pos_weight:.2f}',
    'roc_auc': roc_exp2,
    'pr_auc': pr_exp2,
    'f1_score': f1_exp2,
    'improvement_roc': roc_exp2 - roc_exp1,
    'improvement_pr': pr_exp2 - pr_exp1
}

### Experiment 3: 균형 샘플 앙상블

**왜 이 방법을 썼나?**
- check.ipynb에서 0.9800의 높은 ROC-AUC 달성한 방법
- 클래스 불균형을 근본적으로 해결 (1:10 비율로 재샘플링)
- 여러 샘플로 학습한 모델의 앙상블 → 더 안정적인 예측

**예상 효과:**
- ROC-AUC 크게 향상 (0.96 → 0.97+)
- 모델 안정성 증가
- 단점: 학습 시간 증가 (58개 모델)

In [ ]:
print("\n[Exp 3] 균형 샘플 앙상블")
print("-" * 70)

# 균형 샘플 데이터셋 생성
train_sample_df = pd.concat([X_train, y_train], axis=1)

fraud_train = train_sample_df[train_sample_df['Class'] == 1].copy()
normal_train = train_sample_df[train_sample_df['Class'] == 0].copy()

fraud_count_train = len(fraud_train)
normal_chunk_size = fraud_count_train * 10

print(f"Fraud 개수: {fraud_count_train}")
print(f"한 묶음당 Normal 개수: {normal_chunk_size}")

normal_train = normal_train.sample(frac=1, random_state=42).reset_index(drop=True)

sample_datasets = []
for start in range(0, len(normal_train), normal_chunk_size):
    normal_chunk = normal_train.iloc[start:start + normal_chunk_size]
    if len(normal_chunk) < normal_chunk_size:
        break
    sampled_df = pd.concat([fraud_train, normal_chunk])
    sampled_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)
    sample_datasets.append(sampled_df)

print(f"생성된 균형 샘플: {len(sample_datasets)}개")

# 모델 학습
print("\n모델 학습 중...")
ensemble_models = []
for i, sampled_df in enumerate(sample_datasets):
    X_sample = sampled_df.drop(columns='Class')
    y_sample = sampled_df['Class']

    model = LGBMClassifier(
        n_estimators=300,
        num_leaves=64,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    model.fit(X_sample, y_sample)
    ensemble_models.append(model)
    
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(sample_datasets)} 완료")

print(f"\n학습된 모델: {len(ensemble_models)}개")

# 앙상블 예측
ensemble_pred_proba_list = []
for model in ensemble_models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    ensemble_pred_proba_list.append(pred_proba)

ensemble_mean_proba = np.mean(ensemble_pred_proba_list, axis=0)
y_pred_exp3 = (ensemble_mean_proba >= 0.5).astype(int)

roc_exp3 = roc_auc_score(y_test, ensemble_mean_proba)
pr_exp3 = average_precision_score(y_test, ensemble_mean_proba)
f1_exp3 = f1_score(y_test, y_pred_exp3)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp3:.4f} (Baseline 대비: {roc_exp3-roc_exp1:+.4f}) ⭐⭐")
print(f"  PR-AUC:  {pr_exp3:.4f} (Baseline 대비: {pr_exp3-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp3:.4f} (Baseline 대비: {f1_exp3-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp3))

experiment_results['balanced_ensemble'] = {
    'name': '균형 샘플 앙상블',
    'description': f'Fraud:Normal=1:10 비율의 {len(sample_datasets)}개 샘플 앙상블',
    'roc_auc': roc_exp3,
    'pr_auc': pr_exp3,
    'f1_score': f1_exp3,
    'improvement_roc': roc_exp3 - roc_exp1,
    'improvement_pr': pr_exp3 - pr_exp1
}

### Experiment 4: K-Fold 앙상블

**왜 이 방법을 썼나?**
- 전체 훈련 데이터를 5개 Fold로 나누어 각각 모델 학습
- 모든 데이터를 활용하면서도 과적합 방지
- 앙상블로 인한 예측 안정성 향상

**예상 효과:**
- ROC-AUC 약간 향상
- 모델 안정성 증가
- 학습 시간: 균형 앙상블보다 짧음

In [ ]:
print("\n[Exp 4] K-Fold 앙상블 (5-Fold)")
print("-" * 70)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kfold_models = []
kfold_pred_proba_list = []

print("Fold별 모델 학습 중...")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train), 1):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    model = LGBMClassifier(
        n_estimators=500,
        num_leaves=64,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    model.fit(X_fold_train, y_fold_train)
    kfold_models.append(model)

    # 테스트 예측
    pred_proba = model.predict_proba(X_test)[:, 1]
    kfold_pred_proba_list.append(pred_proba)
    print(f"  Fold {fold}/5 완료")

# 5개 모델 예측의 평균
kfold_mean_proba = np.mean(kfold_pred_proba_list, axis=0)
y_pred_exp4 = (kfold_mean_proba >= 0.5).astype(int)

roc_exp4 = roc_auc_score(y_test, kfold_mean_proba)
pr_exp4 = average_precision_score(y_test, kfold_mean_proba)
f1_exp4 = f1_score(y_test, y_pred_exp4)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp4:.4f} (Baseline 대비: {roc_exp4-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp4:.4f} (Baseline 대비: {pr_exp4-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp4:.4f} (Baseline 대비: {f1_exp4-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp4))

experiment_results['kfold_ensemble'] = {
    'name': 'K-Fold 앙상블',
    'description': '5-Fold 교차검증 기반 앙상블',
    'roc_auc': roc_exp4,
    'pr_auc': pr_exp4,
    'f1_score': f1_exp4,
    'improvement_roc': roc_exp4 - roc_exp1,
    'improvement_pr': pr_exp4 - pr_exp1
}

### Experiment 5: Early Stopping

**왜 이 방법을 썼나?**
- Validation set을 모니터링하며 최적 반복(iteration) 찾기
- 과적합 자동 방지
- 효율성: 불필요한 반복 스킵

**예상 효과:**
- 일반화 성능 향상
- 학습 시간 단축

In [ ]:
print("\n[Exp 5] Early Stopping")
print("-" * 70)

from lightgbm import early_stopping

# Validation set 분리
X_train_es, X_val_es, y_train_es, y_val_es = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"Validation set 분리: {len(X_val_es)}개")

model_exp5 = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    boost_from_average=False,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

# Early Stopping 콜백 명시 (50 라운드 동안 개선 없으면 중단)
model_exp5.fit(
    X_train_es, y_train_es,
    eval_set=[(X_val_es, y_val_es)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_exp5 = model_exp5.predict(X_test)
y_pred_proba_exp5 = model_exp5.predict_proba(X_test)[:, 1]

roc_exp5 = roc_auc_score(y_test, y_pred_proba_exp5)
pr_exp5 = average_precision_score(y_test, y_pred_proba_exp5)
f1_exp5 = f1_score(y_test, y_pred_exp5)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp5:.4f} (Baseline 대비: {roc_exp5-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp5:.4f} (Baseline 대비: {pr_exp5-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp5:.4f} (Baseline 대비: {f1_exp5-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp5))

experiment_results['early_stopping'] = {
    'name': 'Early Stopping',
    'description': 'Validation set 모니터링으로 과적합 방지',
    'roc_auc': roc_exp5,
    'pr_auc': pr_exp5,
    'f1_score': f1_exp5,
    'improvement_roc': roc_exp5 - roc_exp1,
    'improvement_pr': pr_exp5 - pr_exp1
}

### Experiment 6: 임곗값 최적화 (F1-Score 기준)

**왜 이 방법을 썼나?**
- 기본 임곗값 0.5는 불균형 데이터에 최적이 아닐 수 있음
- F1-Score를 최대화하는 임곗값 찾기
- 정밀도(Precision)와 재현율(Recall) 동시 고려

**예상 효과:**
- F1-Score 향상
- 임곗값 변경만으로 성능 개선 (재학습 불필요)

In [ ]:
print("\n[Exp 6] 임곗값 최적화 (F1-Score 기준)")
print("-" * 70)

# Baseline 모델의 예측확률 사용
best_f1 = 0
best_threshold = 0.5
best_pred = y_pred_exp1.copy()

threshold_results = []
for threshold in np.arange(0.1, 0.9, 0.05):
    pred = (y_pred_proba_exp1 >= threshold).astype(int)
    f1 = f1_score(y_test, pred)
    threshold_results.append((threshold, f1))
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_pred = pred

print("임곗값별 F1-Score:")
for threshold, f1 in threshold_results:
    marker = " ← 최적" if threshold == best_threshold else ""
    print(f"  {threshold:.2f}: {f1:.4f}{marker}")

y_pred_exp6 = best_pred

roc_exp6 = roc_auc_score(y_test, y_pred_proba_exp1)  # ROC-AUC는 임곗값과 무관
pr_exp6 = average_precision_score(y_test, y_pred_proba_exp1)
f1_exp6 = best_f1

print(f"\n결과 (최적 임곗값: {best_threshold:.2f}):")
print(f"  ROC-AUC: {roc_exp6:.4f} (Baseline 대비: {roc_exp6-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp6:.4f} (Baseline 대비: {pr_exp6-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp6:.4f} (Baseline 대비: {f1_exp6-f1_exp1:+.4f}) ⭐")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp6))

experiment_results['threshold_opt'] = {
    'name': '임곗값 최적화',
    'description': f'최적 임곗값={best_threshold:.2f} (F1-Score 기준)',
    'roc_auc': roc_exp6,
    'pr_auc': pr_exp6,
    'f1_score': f1_exp6,
    'improvement_f1': f1_exp6 - f1_exp1
}

### 실험 결과 종합 비교

In [ ]:
import pandas as pd

print("\n" + "="*70)
print("실험 결과 종합")
print("="*70)

# 결과 DataFrame 생성
results_data = []
for key, exp in experiment_results.items():
    results_data.append({
        '방법': exp['name'],
        'ROC-AUC': f"{exp['roc_auc']:.4f}",
        'PR-AUC': f"{exp['pr_auc']:.4f}",
        'F1-Score': f"{exp['f1_score']:.4f}"
    })

results_df = pd.DataFrame(results_data)
print("\n" + results_df.to_string(index=False))

# 최고 성능 찾기
best_roc_exp = max(experiment_results.items(), key=lambda x: x[1]['roc_auc'])
best_pr_exp = max(experiment_results.items(), key=lambda x: x[1]['pr_auc'])
best_f1_exp = max(experiment_results.items(), key=lambda x: x[1]['f1_score'])

print("\n" + "="*70)
print("🏆 최고 성능 모델")
print("="*70)
print(f"ROC-AUC 최고: {best_roc_exp[1]['name']} ({best_roc_exp[1]['roc_auc']:.4f})")
print(f"PR-AUC 최고:  {best_pr_exp[1]['name']} ({best_pr_exp[1]['pr_auc']:.4f})")
print(f"F1-Score 최고: {best_f1_exp[1]['name']} ({best_f1_exp[1]['f1_score']:.4f})")

# Baseline 대비 개선도
print("\n" + "="*70)
print("Baseline 대비 개선도")
print("="*70)
for key, exp in experiment_results.items():
    if key != 'baseline':
        if 'improvement_roc' in exp:
            print(f"{exp['name']:20} | ROC-AUC: {exp['improvement_roc']:+.4f}")

### Experiment 7: Hyperopt 기반 하이퍼파라미터 최적화

**왜 이 방법을 썼나?**
- 수동 튜닝의 한계를 극복
- 베이지안 최적화를 통한 효율적인 하이퍼파라미터 탐색
- 6가지 실험의 기본 파라미터를 더 정교하게 조정

**기대 효과:**
- ROC-AUC 추가 개선 (최적 파라미터 조합)
- 모든 실험 중 최고 성능 달성 가능
- 단점: 계산 시간 증가

In [ ]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import cross_val_predict
import numpy as np

print("\n[Exp 7] Hyperopt 기반 하이퍼파라미터 최적화")
print("-" * 70)

# 탐색 공간(Search Space) 정의
space = {
    'num_leaves': hp.choice('num_leaves', [31, 64, 128, 256]),
    'max_depth': hp.choice('max_depth', [5, 7, 10, 15, -1]),
    'min_child_samples': hp.choice('min_child_samples', [10, 20, 30, 50]),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(0.001), np.log(10)),
}

print("탐색 공간 설정 완료")
print("  - num_leaves: [31, 64, 128, 256]")
print("  - max_depth: [5, 7, 10, 15, -1]")
print("  - learning_rate: [0.01 ~ 0.2]")
print("  - subsample: [0.6 ~ 1.0]")
print("  - colsample_bytree: [0.6 ~ 1.0]")
print("  - reg_lambda: [0.001 ~ 10]")

In [ ]:
# 목적 함수(Objective Function) 정의
trial_count = 0
best_score = 0

def objective(params):
    global trial_count, best_score
    trial_count += 1
    
    model = LGBMClassifier(
        **params,
        n_estimators=500,
        boost_from_average=False,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    
    # 3-Fold 층화 교차 검증으로 PR-AUC 평가
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    proba_cv = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    
    score = average_precision_score(y_train, proba_cv)
    
    if score > best_score:
        best_score = score
        print(f"Trial {trial_count:2d} | PR-AUC: {score:.4f} ⭐ 새로운 최고!")
    
    return {'loss': -score, 'status': STATUS_OK}

print("\n목적 함수 정의 완료 (PR-AUC 최대화)")

In [ ]:
# Hyperopt 최적화 실행
print("\nHyperopt 실행 중... (50회 탐색)")
print("시간이 걸릴 수 있습니다. 잠시만 기다려주세요...\n")

trials = Trials()
best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42),
    verbose=0
)

print("\n" + "="*70)
print(f"Hyperopt 탐색 완료!")
print(f"최고 PR-AUC: {best_score:.4f}")
print("="*70)

In [ ]:
# hp.choice 인덱스를 실제 값으로 복원
num_leaves_list = [31, 64, 128, 256]
max_depth_list = [5, 7, 10, 15, -1]
min_child_samples_list = [10, 20, 30, 50]

best_lgbm_params = {
    'num_leaves': num_leaves_list[int(best_params['num_leaves'])],
    'max_depth': max_depth_list[int(best_params['max_depth'])],
    'min_child_samples': min_child_samples_list[int(best_params['min_child_samples'])],
    'learning_rate': best_params['learning_rate'],
    'subsample': best_params['subsample'],
    'colsample_bytree': best_params['colsample_bytree'],
    'reg_lambda': best_params['reg_lambda'],
    'n_estimators': 1000,
    'boost_from_average': False,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

print("\n최적 하이퍼파라미터:")
print("-" * 70)
for key, value in best_lgbm_params.items():
    if isinstance(value, float):
        print(f"  {key:25} = {value:.6f}")
    else:
        print(f"  {key:25} = {value}")

In [ ]:
# 최적 파라미터로 모델 학습
print("\n최적 파라미터로 모델 학습 중...")

tuned_lgbm = LGBMClassifier(**best_lgbm_params)
tuned_lgbm.fit(X_train, y_train)

# Test 데이터 예측
y_pred_exp7 = tuned_lgbm.predict(X_test)
y_pred_proba_exp7 = tuned_lgbm.predict_proba(X_test)[:, 1]

# 성능 평가
roc_exp7 = roc_auc_score(y_test, y_pred_proba_exp7)
pr_exp7 = average_precision_score(y_test, y_pred_proba_exp7)
f1_exp7 = f1_score(y_test, y_pred_exp7)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp7:.4f} (Baseline 대비: {roc_exp7-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp7:.4f} (Baseline 대비: {pr_exp7-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp7:.4f} (Baseline 대비: {f1_exp7-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp7))

experiment_results['hyperopt'] = {
    'name': 'Hyperopt 최적화',
    'description': f'베이지안 최적화 (50회 탐색)',
    'roc_auc': roc_exp7,
    'pr_auc': pr_exp7,
    'f1_score': f1_exp7,
    'improvement_roc': roc_exp7 - roc_exp1,
    'improvement_pr': pr_exp7 - pr_exp1,
    'best_params': best_lgbm_params
}

### 최종 종합: 모든 7가지 실험 비교

In [ ]:
print("\n[Exp 5] Early Stopping")
print("-" * 70)

from lightgbm import early_stopping

# Validation set 분리
X_train_es, X_val_es, y_train_es, y_val_es = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"Validation set 분리: {len(X_val_es)}개")

model_exp5 = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    boost_from_average=False,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

# Early Stopping 콜백 명시 (50 라운드 동안 개선 없으면 중단)
model_exp5.fit(
    X_train_es, y_train_es,
    eval_set=[(X_val_es, y_val_es)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_exp5 = model_exp5.predict(X_test)
y_pred_proba_exp5 = model_exp5.predict_proba(X_test)[:, 1]

roc_exp5 = roc_auc_score(y_test, y_pred_proba_exp5)
pr_exp5 = average_precision_score(y_test, y_pred_proba_exp5)
f1_exp5 = f1_score(y_test, y_pred_exp5)

print(f"\n결과:")
print(f"  ROC-AUC: {roc_exp5:.4f} (Baseline 대비: {roc_exp5-roc_exp1:+.4f})")
print(f"  PR-AUC:  {pr_exp5:.4f} (Baseline 대비: {pr_exp5-pr_exp1:+.4f})")
print(f"  F1-Score: {f1_exp5:.4f} (Baseline 대비: {f1_exp5-f1_exp1:+.4f})")
print(f"\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred_exp5))

experiment_results['early_stopping'] = {
    'name': 'Early Stopping',
    'description': 'Validation set 모니터링으로 과적합 방지',
    'roc_auc': roc_exp5,
    'pr_auc': pr_exp5,
    'f1_score': f1_exp5,
    'improvement_roc': roc_exp5 - roc_exp1,
    'improvement_pr': pr_exp5 - pr_exp1
}

### 📊 최종 결론: 기존 Hyperopt가 최고 성능

#### 🏆 성능 순위

| 순위 | 방법 | ROC-AUC | F1-Score | 평가 |
|------|------|---------|----------|------|
| 🥇 | **기존 Hyperopt (처음)** | **0.9687** | 0.8690 | ✅ 최고 성능 |
| 🥈 | 균형 샘플 앙상블 | 0.9705 | 0.6556 | 높은 ROC, 낮은 F1 |
| 🥉 | Baseline (기본 모델) | 0.9681 | 0.8757 | 안정적 |
| 4️⃣ | 임곗값 최적화 | 0.9681 | **0.8824** | 실무 추천 |
| 5️⃣ | 클래스 가중치 | 0.9667 | 0.8623 | 성능 저하 |
| ❌ | Early Stopping | 0.7256 | 0.5600 | **-24.3%** |
| ❌ | K-Fold 앙상블 | 0.8684 | 0.6014 | **-9.7%** |

---

#### 💡 핵심 발견

1. **기존 Hyperopt (0.9687)** = 6가지 새로운 실험의 **최고 성능 달성**
   - Baseline 대비: +0.06%
   - 균형 샘플 대비: -0.18% (하지만 F1은 더 안정적)

2. **왜 기존 Hyperopt가 우수했나?**
   - ✅ 적절한 탐색 공간 설계 (num_leaves: 3개, max_depth: 4개)
   - ✅ 50회 탐색으로 충분히 수렴
   - ✅ 과도한 정규화 없음 (reg_lambda 미포함)
   - ✅ 불균형 데이터에 맞는 단순함

3. **새로운 시도들이 실패한 이유**
   - ❌ K-Fold: 사기 샘플이 너무 적어짐 (373개 → 75개/fold)
   - ❌ Early Stopping: 검증 데이터 분리로 훈련 데이터 부족 (226K → 145K)
   - ❌ 클래스 가중치: scale_pos_weight=607.51 과도함
   - ❌ 균형 샘플: F1-Score 극단적 저하 (0.88 → 0.66)

---

#### 🎯 최종 권장사항

##### 프로덕션 배포
```python
## 기존 Hyperopt 모델 사용
LGBMClassifier(
    num_leaves=128,
    max_depth=5,
    learning_rate=0.0476,
    subsample=0.8353,
    colsample_bytree=0.6040,
    n_estimators=1000
)
## ROC-AUC: 0.9687 ⭐⭐⭐
```

##### 빠른 개선 필요시
```python
## 임곗값 최적화 (재학습 불필요)
threshold = 0.10  # 0.5 → 0.10
## F1-Score: 0.8824 ✅
```

---

#### 📌 교훈

**"복잡한 방법이 항상 좋은 건 아니다"**

- 데이터가 작고 불균형할 때는 단순한 최적화가 최고
- 베이지안 최적화의 탐색 공간 설계가 가장 중요
- K-Fold, Early Stopping은 대규모 데이터용 기법
- 한 가지 지표(ROC-AUC)만으로 판단하면 안 됨
- **기존 하이퍼파라미터가 최고의 선택이었다** ✅

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#4C72B0', '#C44E52']
counts = train_df['Class'].value_counts()

bars = ax.bar(['Normal (0)', 'Fraud (1)'], counts.values, color=colors, width=0.5)
ax.set_yscale('log')
ax.set_title('Class Imbalance Distribution (Log Scale)', fontsize=13, pad=12)
ax.set_ylabel('Transaction Count (Log)', fontsize=11)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:,}\n({yval/sum(counts)*100:.2f}%)', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='Class', y='V14', data=train_df, palette=['#4C72B0', '#C44E52'], ax=ax, width=0.4)

# 사기 데이터 기준 IQR 1.5 하단 경계선 표시
fraud_v14 = train_df[train_df['Class'] == 1]['V14']
q25 = fraud_v14.quantile(0.25)
iqr = fraud_v14.quantile(0.75) - q25
threshold_v14 = q25 - (iqr * 1.5)

ax.axhline(threshold_v14, color='red', linestyle='--', label=f'Outlier Threshold ({threshold_v14:.2f})')
ax.set_title('V14 Feature Distribution & Outlier Detection', fontsize=12, pad=10)
ax.set_xlabel('Class', fontsize=10)
ax.set_ylabel('V14 Value', fontsize=10)
ax.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
# Class와의 절대값 상관계수 상위 8개 추출
corr = train_df.corr()
top_corr_features = corr['Class'].abs().sort_values(ascending=False).head(9).index

plt.figure(figsize=(8, 6))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Top Correlated Features with Class', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 스케일링 전 원본 Amount
sns.kdeplot(train_df['Amount'], ax=axes[0], fill=True, color='#C44E52')
axes[0].set_title('Original Amount Distribution (Raw)', fontsize=12)
axes[0].set_xlabel('Amount ($)')
axes[0].set_xlim(0, 2000)

# 스케일링 후 Amount_Scaled (train_df_processed 기준)
sns.kdeplot(train_df_processed['Amount_Scaled'], ax=axes[1], fill=True, color='#4C72B0')
axes[1].set_title('RobustScaled Amount Distribution', fontsize=12)
axes[1].set_xlabel('Amount_Scaled')
axes[1].set_xlim(-5, 20)

plt.tight_layout()
plt.show()

## [이진희] XGBoost

앞부분(중복 제거 전 학습 → train/test 겹침 확인)은 데이터 누수를 발견한 과정 그대로 남겨 두었다. 이후 `drop_duplicates()` 를 넣어 다시 진행한다.

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드



# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




In [ ]:
# 트레인 

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain Fraud ratio:")
print(y_train.value_counts(normalize=True))

print("\nTest Fraud ratio:")
print(y_test.value_counts(normalize=True))

In [ ]:
train_check = X_train.copy()
test_check = X_test.copy()

train_rows = set(
    map(tuple, train_check.to_numpy())
)

test_rows = set(
    map(tuple, test_check.to_numpy())
)

overlap = train_rows & test_rows

print(
    "Train/Test 완전 동일 Feature 행:",
    len(overlap)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드

print("중복 제거 전:", len(train_df))

train_df = train_df.drop_duplicates().reset_index(drop=True)

print("중복 제거 후:", len(train_df))

# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




In [ ]:
#트레인 

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain Fraud ratio:")
print(y_train.value_counts(normalize=True))

print("\nTest Fraud ratio:")
print(y_test.value_counts(normalize=True))

In [ ]:
train_check = X_train.copy()
test_check = X_test.copy()

train_rows = set(
    map(tuple, train_check.to_numpy())
)

test_rows = set(
    map(tuple, test_check.to_numpy())
)

overlap = train_rows & test_rows

print(
    "Train/Test 완전 동일 Feature 행:",
    len(overlap)
)

In [ ]:
# amount scaling
from sklearn.preprocessing import StandardScaler, RobustScaler

X_train = X_train.copy()
X_test = X_test.copy()

rob_scaler = RobustScaler()

# Train 데이터에서만 중앙값/IQR 학습
X_train['Amount_Scaled'] = rob_scaler.fit_transform(
    X_train[['Amount']]
)

# Test는 Train에서 배운 기준으로 transform만
X_test['Amount_Scaled'] = rob_scaler.transform(
    X_test[['Amount']]
)

# 기존 Time, Amount 제거
X_train.drop(columns=['Time', 'Amount'], inplace=True)
X_test.drop(columns=['Time', 'Amount'], inplace=True)

In [ ]:

#이상치 제거

def get_outlier(df, column, weight=1.5):

    fraud = df[df['Class'] == 1][column]

    q25 = np.percentile(fraud.values, 25)
    q75 = np.percentile(fraud.values, 75)

    iqr = q75 - q25

    lowest = q25 - weight * iqr
    highest = q75 + weight * iqr

    outlier_index = fraud[
        (fraud < lowest) |
        (fraud > highest)
    ].index

    return outlier_index

In [ ]:
# 데이터 합침.
train_df = X_train.copy()
train_df['Class'] = y_train

In [ ]:
outlier_index = get_outlier(
    train_df,
    column='V14',
    weight=1.5
)

print("제거되는 Train 이상치:", len(outlier_index))

train_df.drop(
    index=outlier_index,
    inplace=True
)

In [ ]:
#사기 / 일반 분리
fraud_train = train_df[
    train_df['Class'] == 1
].copy()

normal_train = train_df[
    train_df['Class'] == 0
].copy()

fraud_count = len(fraud_train)

print("Fraud Train:", fraud_count)
print("Normal Train:", len(normal_train))

In [ ]:
normal_chunk_size = fraud_count * 10

normal_train = normal_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

sample_datasets = []

for start in range(
    0,
    len(normal_train),
    normal_chunk_size
):

    normal_chunk = normal_train.iloc[
        start:start + normal_chunk_size
    ]

    if len(normal_chunk) < normal_chunk_size:
        break

    sampled_df = pd.concat([
        fraud_train,
        normal_chunk
    ])

    sampled_df = sampled_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    sample_datasets.append(sampled_df)

print(
    "만들어진 Sample 수:",
    len(sample_datasets)
)

In [ ]:
xgb_models = []

for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(
        columns='Class'
    )

    y_sample = sampled_df['Class']

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_sample,
        y_sample
    )

    xgb_models.append(model)

print(
    "학습된 XGB 모델:",
    len(xgb_models)
)

In [ ]:
#테스트는 예측으로 수행
xgb_pred_proba_list = []

for model in xgb_models:

    pred_proba = model.predict_proba(
        X_test
    )[:, 1]

    xgb_pred_proba_list.append(
        pred_proba
    )

In [ ]:
#최종 프로우드 확률
xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [ ]:
from sklearn.metrics import average_precision_score


roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

pr_auc = average_precision_score(
    y_test,
    xgb_mean_pred_proba
)

print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(
    confusion_matrix(
        y_test,
        final_pred
    )
)

print(
    classification_report(
        y_test,
        final_pred
    )
)

In [ ]:
# hyperopt 하이퍼 파라미터 튜닝

from hyperopt import hp


# max_depth는 4에서 15까지 1간격으로, min_child_weight는 1에서 6까지 1간격으로
# colsample_bytree는 0.5에서 0.95사이, learning_rate는 0.01에서 0.2사이 정규 분포된 값으로 검색.


xgb_search_space = {'max_depth': hp.quniform('max_depth', 4, 15, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.95),
                    'subsample': hp.uniform('subsample', 0.6, 1.0),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}
# model = XGBClassifier(
#         n_estimators=300,
#         max_depth=5,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         eval_metric='auc',
#         random_state=42,
#         n_jobs=-1
#     )


In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


# 목적 함수 설정.
# 추후 fmin()에서 입력된 search_space값으로 XGBClassifier 교차 검증 학습 후 -1* roc_auc 평균 값을 반환.  
def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            colsample_bytree=search_space['colsample_bytree'],
                            subsample=search_space['subsample'],
                            early_stopping_rounds=30, eval_metric='auc'
                           )

    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list= []
   
    # 3개 k-fold방식 적용
    kf = KFold(n_splits=5)
    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]
        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        xgb_clf.fit(X_tr, y_tr, 
                   eval_set=[(X_tr, y_tr), (X_val, y_val)])
   
        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)
       
    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)


In [ ]:
from hyperopt import fmin, tpe, Trials


trials = Trials()


# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=100, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))


print('best:', best)

In [ ]:
for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(
        columns='Class'
    )

    y_sample = sampled_df['Class']

    # model = XGBClassifier(
    #     n_estimators=300,
    #     max_depth=5,
    #     learning_rate=0.05,
    #     subsample=0.8,
    #     colsample_bytree=0.8,
    #     eval_metric='auc',
    #     random_state=42,
    #     n_jobs=-1
    # )
    model= XGBClassifier(n_estimators=1000, learning_rate=round(best['learning_rate'], 5),
                        max_depth=int(best['max_depth']), min_child_weight=int(best['min_child_weight']),
                        eval_metric="auc",subsample=best['subsample'],
                        colsample_bytree=round(best['colsample_bytree'], 5),random_state=42,
                       n_jobs=-1  
                       )

    model.fit(
        X_sample,
        y_sample
    )

    xgb_models.append(model)

In [ ]:
#테스트는 예측으로 수행
xgb_pred_proba_list = []

for model in xgb_models:

    pred_proba = model.predict_proba(
        X_test
    )[:, 1]

    xgb_pred_proba_list.append(
        pred_proba
    )

#최종 프로우드 확률
xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [ ]:
roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

pr_auc = average_precision_score(
    y_test,
    xgb_mean_pred_proba
)

print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)

In [ ]:
final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(
    confusion_matrix(
        y_test,
        final_pred
    )
)

print(
    classification_report(
        y_test,
        final_pred
    )
)

### [이진희] 5-Fold 교차검증 (추가 검증)

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score

from xgboost import XGBClassifier

# 데이터 로드

print("중복 제거 전:", len(train_df))

train_df = train_df.drop_duplicates().reset_index(drop=True)

print("중복 제거 후:", len(train_df))

# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

roc_scores = []
ap_scores = []

In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_features, y_labels), start=1):

    print(f"\n===== Fold {fold} =====")

    X_train = X_features.iloc[train_idx].copy()
    X_val = X_features.iloc[val_idx].copy()

    y_train = y_labels.iloc[train_idx].copy()
    y_val = y_labels.iloc[val_idx].copy()

    # -------------------------
    # 1. Scaling
    # -------------------------
    scaler = RobustScaler()

    X_train['Amount_Scaled'] = scaler.fit_transform(
        X_train[['Amount']]
    )

    X_val['Amount_Scaled'] = scaler.transform(
        X_val[['Amount']]
    )

    X_train.drop(columns=['Time', 'Amount'], inplace=True)
    X_val.drop(columns=['Time', 'Amount'], inplace=True)

    # -------------------------
    # 2. Train 데이터 다시 결합
    # -------------------------
    train_df = X_train.copy()
    train_df['Class'] = y_train

    # -------------------------
    # 3. V14 이상치 제거
    # -------------------------
    fraud_v14 = train_df[
        train_df['Class'] == 1
    ]['V14']

    q25 = np.percentile(fraud_v14, 25)
    q75 = np.percentile(fraud_v14, 75)

    iqr = q75 - q25

    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr

    outlier_index = fraud_v14[
        (fraud_v14 < lower) |
        (fraud_v14 > upper)
    ].index

    train_df.drop(
        index=outlier_index,
        inplace=True
    )

    # -------------------------
    # 4. Fraud / Normal 분리
    # -------------------------
    fraud_train = train_df[
        train_df['Class'] == 1
    ].copy()

    normal_train = train_df[
        train_df['Class'] == 0
    ].copy()

    fraud_count = len(fraud_train)

    chunk_size = fraud_count * 10

    normal_train = normal_train.sample(
        frac=1,
        random_state=42 + fold
    ).reset_index(drop=True)

    # -------------------------
    # 5. 여러 10:1 샘플 생성
    # -------------------------
    sample_datasets = []

    for start in range(
        0,
        len(normal_train),
        chunk_size
    ):

        normal_chunk = normal_train.iloc[
            start:start + chunk_size
        ]

        if len(normal_chunk) < chunk_size:
            break

        sampled_df = pd.concat([
            fraud_train,
            normal_chunk
        ])

        sampled_df = sampled_df.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        sample_datasets.append(sampled_df)

    print("샘플 수:", len(sample_datasets))

    # -------------------------
    # 6. XGB 여러 개 학습
    # -------------------------
    pred_proba_list = []

    for i, sampled_df in enumerate(sample_datasets):

        X_sample = sampled_df.drop(
            columns='Class'
        )

        y_sample = sampled_df['Class']

        model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            random_state=42 + i,
            n_jobs=-1
        )

        model.fit(
            X_sample,
            y_sample
        )

        pred_proba = model.predict_proba(
            X_val
        )[:, 1]

        pred_proba_list.append(
            pred_proba
        )

    # -------------------------
    # 7. 앙상블 평균
    # -------------------------
    mean_pred_proba = np.mean(
        pred_proba_list,
        axis=0
    )

    # -------------------------
    # 8. Fold 평가
    # -------------------------
    roc = roc_auc_score(
        y_val,
        mean_pred_proba
    )

    ap = average_precision_score(
        y_val,
        mean_pred_proba
    )

    roc_scores.append(roc)
    ap_scores.append(ap)

    print("ROC-AUC:", roc)
    print("AP:", ap)

In [ ]:
print("\n==============================")
print("5-Fold 결과")
print("==============================")

print("ROC-AUC scores:")
print(roc_scores)

print(
    "ROC-AUC Mean:",
    np.mean(roc_scores)
)

print(
    "ROC-AUC Std:",
    np.std(roc_scores)
)

print("\nAP scores:")
print(ap_scores)

print(
    "AP Mean:",
    np.mean(ap_scores)
)

print(
    "AP Std:",
    np.std(ap_scores)
)

## [김소현] Logistic Regression

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(1234)


In [ ]:
# 데이터 로드

print("Length:", len(train_df))
train_df.head()


In [ ]:
train_df.shape
# 총 284,807행, 31개 컬럼 (Time, V1~V28, Amount, Class)


In [ ]:
train_df.info()
# 전부 float64 / int64 -> 문자형(object) 컬럼 없음
# Non-Null Count가 전부 284807이면 결측치가 없다는 뜻 -> 아래에서 isnull()로 재확인


In [ ]:
# PCA 처리된 컬럼 확인 (V1~V28)
# 근거: ① 평균이 거의 0  ② 서로 상관관계가 거의 없음(직교) -> PCA 변환의 전형적인 특징
v_cols = [c for c in train_df.columns if c.startswith("V")]

print("V1~V28 평균 절대값 최대:", train_df[v_cols].mean().abs().max())
print("V1~V28 상관계수 절대값 최대(자기 자신 제외):",
      train_df[v_cols].corr().where(~np.eye(len(v_cols), dtype=bool)).abs().max().max())

# => V1~V28(28개)만 PCA로 변환된 익명 피처, Time/Amount는 원본 컬럼, Class는 타겟
#
# [PCA 컬럼이 이후 전처리/모델링에 미치는 영향]
# - 이미 평균 0이지만 컬럼별 표준편차는 서로 다름(V1≈1.95 ~ V28≈0.33, explained variance 순서)
#   -> 로지스틱회귀/KNN/SVM처럼 스케일에 민감한 모델을 쓴다면 StandardScaler로 한 번 더 맞춰주는 게 안전
#   -> RandomForest/XGBoost/LightGBM 같은 트리 기반 모델은 스케일 영향을 안 받아 그대로 사용 가능
# - PCA 성분끼리는 이미 직교(무상관)라서 V1~V28 사이의 다중공선성(multicollinearity)은 걱정할 필요 없음
# - 이미 익명화된 조합 변수라 원래 의미(어떤 거래 속성인지)를 알 수 없어, 개별 성분 단위의 도메인 기반
#   피처 엔지니어링은 불가능함 (feature importance로 "V14가 중요하다" 정도만 알 수 있고 비즈니스 해석은 불가)


In [ ]:
train_df.describe()
# V1~V28은 PCA로 변환된 익명 피처라서 평균이 거의 0에 가까움
# -> mean 값이 3.919560e-15 처럼 지수(e) 표기로 출력되어 한눈에 비교하기 어려움


In [ ]:
# 지수(e) 표기 처리
# 기본 pandas 출력은 아주 작은 값(예: e-15)을 지수 표기로 보여줘서 가독성이 떨어짐
# -> 소수점 표기로 통일해서 보기 쉽게 설정
pd.set_option("display.float_format", lambda x: "%.4f" % x)

train_df.describe()
# 위와 동일한 값이지만 e-15 같은 지수 표기 없이 0.0000 형태로 표시됨


In [ ]:
# 타겟(Class) 분포 확인
print(train_df["Class"].value_counts())
print()
print(train_df["Class"].value_counts(normalize=True))

# Class = 0 (정상 거래) : 약 99.83%
# Class = 1 (사기 거래) : 약 0.17%  -> 극단적인 클래스 불균형


In [ ]:
# Class 분포 시각화
import matplotlib.pyplot as plt

# 그래프에 한글이 깨지지 않도록 폰트 설정 (Windows 기본 한글 폰트)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

class_counts = train_df["Class"].value_counts()

plt.figure(figsize=(5, 4))
plt.bar(["정상(0)", "사기(1)"], class_counts.values, color=["skyblue", "orange"])
plt.title("Class 분포 (정상 vs 사기)")
plt.ylabel("건수")
for i, v in enumerate(class_counts.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()


In [ ]:
# 결측치 확인
print("컬럼별 결측치 개수:")
print(train_df.isnull().sum())

print("\n결측치가 있는 컬럼 수:", (train_df.isnull().sum() > 0).sum())
# 결측치 없음 확인 -> 별도의 결측치 대체(imputation)는 불필요


In [ ]:
# Time 컬럼 확인 및 제거
# Time은 첫 거래 시점부터 누적된 경과 초(0~172,792)일 뿐, 실제 시각(시/분)이 아니라서
# 그 자체로는 사기 여부와 직접적인 관계를 갖는 피처로 보기 어려움 + 값이 시간순으로 계속 증가하므로
# 그대로 학습에 쓰면 특정 시간대에 과적합될 위험도 있음
# -> 중복행 확인/제거에 들어가기 전에 먼저 Time 컬럼을 제거
#    (Time을 남긴 채로 중복을 제거하면, Time만 다르고 나머지 30개 컬럼이 같은 행을
#     서로 다른 행으로 잘못 취급해서 중복행이 과소 집계됨)
print(train_df["Time"].describe())

train_df = train_df.drop(columns=["Time"])
print("\nTime 컬럼 제거 후 shape:", train_df.shape)


In [ ]:
# Feature-Class 상관관계 확인 - 어떤 컬럼이 사기 여부와 관련이 깊은지 파악
corr_with_class = train_df.corr()["Class"].drop("Class").sort_values()

plt.figure(figsize=(6, 9))
colors = ["crimson" if v < 0 else "steelblue" for v in corr_with_class.values]
plt.barh(corr_with_class.index, corr_with_class.values, color=colors)
plt.title("각 Feature와 Class의 상관계수")
plt.xlabel("상관계수")
plt.tight_layout()
plt.show()

print(corr_with_class)
# V14, V17, V12, V10, V4, V11 등이 Class와 상관관계가 가장 강한 컬럼들
# -> 이 중 V14를 이상치 점검 대상으로 살펴봄


In [ ]:
# V14 분포 확인 - Class별 boxplot (이상치가 어디 몰려있는지 시각적으로 확인)
train_df.boxplot(column="V14", by="Class", figsize=(6, 4))
plt.title("V14 분포 (Class별)")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("V14")
plt.show()
# Class=1(사기)쪽에 아래로 크게 벗어난 극단값들이 보임 -> 아래 셀에서 이상치로 확인/제거


In [ ]:
# V14 이상치 제거 (사기 케이스 기준)
# 사기(Class=1) 케이스만 떼어서 V14의 IQR을 구하고, 하한(Q1 - 1.5*IQR) 아래로 떨어지는 값만 제거
# (V14가 Class와 상관관계가 가장 큰 컬럼 중 하나라, 극단적으로 벗어난 값이 실제 사기 패턴이라기보다
#  노이즈에 가까울 가능성이 있어 점검함 - 정상(Class=0) 쪽은 건드리지 않음)
# * 반드시 중복행 제거보다 먼저 함 - 중복 제거 후에는 사분위수 자체가 바뀌어서 같은 기준이라도 결과가 달라짐
fraud_v14 = train_df.loc[train_df["Class"] == 1, "V14"]
q1, q3 = fraud_v14.quantile(0.25), fraud_v14.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr

v14_outliers = train_df[(train_df["Class"] == 1) & (train_df["V14"] < lower_bound)]
print("V14 하한(사기 케이스 기준):", lower_bound)
print("제거 대상 행 수:", len(v14_outliers))
print(v14_outliers[["V14", "Class"]])

train_df = train_df.drop(index=v14_outliers.index)
print("\nV14 이상치 제거 후 shape:", train_df.shape)


In [ ]:
# 중복행 확인 및 제거
dup_cnt = train_df.duplicated().sum()
print("중복행 개수:", dup_cnt)

train_df = train_df.drop_duplicates().reset_index(drop=True)
print("중복 제거 후 shape:", train_df.shape)

print("\n중복 제거 후 Class 분포:")
print(train_df["Class"].value_counts())


In [ ]:
# Amount 이상치(너무 높은/낮은 값) 확인 - min/max 기준
print("Amount 최소값:", train_df["Amount"].min())
print("Amount 최대값:", train_df["Amount"].max())

print("\nAmount가 가장 큰 상위 10건:")
print(train_df.sort_values("Amount", ascending=False)[["Amount", "Class"]].head(10))

print("\nClass별 Amount 통계:")
print(train_df.groupby("Class")["Amount"].describe())

# min = 0, max ≈ 25,691 -> 둘 다 실제로 발생 가능한 결제 금액대라 값 자체가 잘못 기록된 오류로 보기 어려움
# 최고액 상위 10건은 전부 Class=0(정상) 거래이고, Class=1(사기)의 max는 2,125.87로 오히려 훨씬 작음
# -> IQR 기준 "이상치"로 잡히는 고액 거래들은 실제로는 정상적인 고액 결제이며,
#    이 값들을 clip/제거하면 모델이 학습할 수 있는 금액 스케일 정보만 없어질 뿐 사기 탐지에는 도움이 안 됨
# => Amount 이상치 제거/클리핑은 불필요하다고 판단, 원본 값을 그대로 사용


In [ ]:
# Amount 분포 시각화 - 왜곡(skew) 정도를 원본/log 스케일로 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_df["Amount"], bins=50, color="skyblue")
axes[0].set_title("Amount 분포 (원본 스케일)")
axes[0].set_xlabel("Amount")

axes[1].hist(np.log1p(train_df["Amount"]), bins=50, color="salmon")
axes[1].set_title("Amount 분포 (log1p 변환)")
axes[1].set_xlabel("log1p(Amount)")

plt.tight_layout()
plt.show()
# 원본 스케일은 대부분 0 근처에 몰려있고 꼬리가 아주 긺 -> log 변환하면 훨씬 정규분포에 가까워짐
# (다만 앞서 결정한 대로 Amount 자체는 클리핑/변환 없이 원본 값을 그대로 사용)


In [ ]:
# 전처리 결과 최종 확인
print("전처리 완료 후 shape:", train_df.shape)
print("결측치 총합:", train_df.isnull().sum().sum())
print("중복행 총합:", train_df.duplicated().sum())

train_df.head()


In [ ]:
# 전처리 완료 후 Class 분포 시각화 (V14 이상치 제거 + 중복행 제거 반영된 최종 상태)
final_class_counts = train_df["Class"].value_counts()

plt.figure(figsize=(5, 4))
plt.bar(["정상(0)", "사기(1)"], final_class_counts.values, color=["skyblue", "orange"])
plt.title("전처리 완료 후 Class 분포")
plt.ylabel("건수")
for i, v in enumerate(final_class_counts.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()

print("원본 Class 분포     :", "정상 284,315 / 사기 492")
print(f"전처리 후 Class 분포 : 정상 {final_class_counts[0]:,} / 사기 {final_class_counts[1]:,}")


In [ ]:
# 전처리 완료된 feature들의 상관관계 heatmap - 전체 관계를 한눈에 확인
feature_cols = train_df.columns
corr_matrix = train_df[feature_cols].corr()

plt.figure(figsize=(12, 10))
im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, label="상관계수")
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title("Feature 상관관계 Heatmap")
plt.tight_layout()
plt.show()

# V1~V28끼리는 거의 하얀색(무상관)인 게 PCA의 직교성과 일치
# Amount/Class만 스케일이 달라 상대적으로 눈에 띔


In [ ]:
# Feature / Target 분리
X = train_df.drop(columns=["Class"])
y = train_df["Class"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# train/test 분리 - stratify로 극단적인 클래스 불균형(양성 0.17%)이 train/test에 동일 비율로 유지되게 함
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

print("\ntrain Class 비율:")
print(y_train.value_counts(normalize=True))

print("\ntest Class 비율:")
print(y_test.value_counts(normalize=True))


In [ ]:
# 스케일링 - RobustScaler (V1~V28 + Amount 전체)
# Amount는 mean이 median의 4배(88 vs 22)일 정도로 심하게 skew돼 있고, 그 이상치를 일부러 유지하기로 했음
# -> 평균/표준편차 기반 StandardScaler는 그 소수의 이상치에 스케일 계산 자체가 끌려가 나머지 값들이 좁은 구간에 눌림
# -> 중앙값/IQR 기반이라 이상치에 덜 흔들리는 RobustScaler를 V1~V28 + Amount 전체에 통일해서 적용
# (반드시 train/test split 이후, train으로만 fit -> test는 transform만 해서 데이터 누수 방지)
# * 이 스케일된 버전은 SMOTE + LogisticRegression처럼 스케일에 민감한 모델 트랙에서만 사용
#   XGBoost/LightGBM 등 트리 기반 모델은 스케일 영향이 없으므로 원본 X_train/X_test를 그대로 사용
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("X_train_scaled shape:", X_train_scaled.shape)
X_train_scaled.describe()


In [ ]:
# 모든 모델 비교/검증에 공통으로 재사용할 StratifiedKFold
# -> 어떤 모델이든 같은 fold 구성으로 평가해야 "모델 차이"를 공정하게 비교할 수 있음
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


In [ ]:
# LogisticRegression (class_weight="balanced") - StratifiedKFold CV
# SMOTE 없이, 이미 전체로 스케일링해둔 X_train_scaled를 그대로 사용
# (스케일링을 fold마다 다시 안 하는 건 SMOTE와 달리 새 데이터를 만드는 게 아니라서 leak이 사실상 무시할 수준)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(
    LogisticRegression(class_weight="balanced", random_state=0),
    X_train_scaled, y_train,
    cv=skf, scoring="roc_auc"
)

print("fold별 ROC-AUC:", lr_cv_scores)
print(f"CV ROC-AUC: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")


In [ ]:
# LogisticRegression + SMOTE - StratifiedKFold CV
# 스케일링은 이미 끝난 X_train_scaled를 그대로 쓰고(leak 무시 가능 수준), SMOTE만 fold마다 새로 실행되게
# imblearn Pipeline에 SMOTE + 모델 딱 2단계만 묶음 (스케일러는 다시 안 넣어도 됨)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

smote_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=0)),
    ("clf", LogisticRegression(random_state=0))
])

smote_cv_scores = cross_val_score(smote_pipeline, X_train_scaled, y_train, cv=skf, scoring="roc_auc")

print("fold별 ROC-AUC:", smote_cv_scores)
print(f"CV ROC-AUC: {smote_cv_scores.mean():.4f} ± {smote_cv_scores.std():.4f}")


In [ ]:
# class_weight vs SMOTE 비교 -> 이긴 쪽을 LogisticRegression의 최종 방식으로 채택
print(f"class_weight : {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")
print(f"SMOTE        : {smote_cv_scores.mean():.4f} ± {smote_cv_scores.std():.4f}")

use_smote = smote_cv_scores.mean() >= lr_cv_scores.mean()
print(f"\n채택: {'SMOTE' if use_smote else 'class_weight=balanced'}")


In [ ]:
# LogisticRegression 하이퍼파라미터 튜닝 - C(규제 강도) GridSearchCV
# 지금까지는 기본값 C=1.0만 썼음 -> 채택된 class_weight="balanced" 방식으로 C만 탐색 (같은 skf 사용)
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    LogisticRegression(class_weight="balanced", random_state=0, max_iter=1000),
    param_grid,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print("C별 CV ROC-AUC:")
for c, score in zip(param_grid["C"], grid_search.cv_results_["mean_test_score"]):
    print(f"  C={c}: {score:.4f}")

print(f"\n최적 C: {grid_search.best_params_['C']}")
print(f"최적 CV ROC-AUC: {grid_search.best_score_:.4f}  (튜닝 전 기본 C=1.0: {lr_cv_scores.mean():.4f})")


In [ ]:
# 최종 재학습 - 채택된 방식 + 튜닝된 C로 X_train_scaled 전체를 다시 학습 -> X_test_scaled로 딱 한 번 평가
if use_smote:
    smote_final = SMOTE(random_state=0)
    X_train_final, y_train_final = smote_final.fit_resample(X_train_scaled, y_train)
    lr_final = LogisticRegression(C=grid_search.best_params_["C"], random_state=0, max_iter=1000)
else:
    X_train_final, y_train_final = X_train_scaled, y_train
    lr_final = LogisticRegression(
        class_weight="balanced",
        C=grid_search.best_params_["C"],
        random_state=0,
        max_iter=1000
    )

lr_final.fit(X_train_final, y_train_final)

lr_final_test_roc = roc_auc_score(y_test, lr_final.predict_proba(X_test_scaled)[:, 1])
method_name = "SMOTE" if use_smote else "class_weight=balanced"
print(f"LogisticRegression ({method_name}, C={grid_search.best_params_['C']}) CV ROC-AUC:   {grid_search.best_score_:.4f}")
print(f"LogisticRegression ({method_name}, C={grid_search.best_params_['C']}) Test ROC-AUC: {lr_final_test_roc:.4f}")


In [ ]:
# pickle로 저장 - 이후 다른 모델(XGBoost/LightGBM 등)도 같은 skf로 평가해서 같은 방식으로 저장 예정
import pickle
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

with open(model_dir / "lr_final.pkl", "wb") as f:
    pickle.dump(
        {
            "model": lr_final,
            "scaler": scaler,
            "method": method_name,
            "best_C": grid_search.best_params_["C"],
            "cv_roc_auc_class_weight_mean": lr_cv_scores.mean(),
            "cv_roc_auc_class_weight_std": lr_cv_scores.std(),
            "cv_roc_auc_smote_mean": smote_cv_scores.mean(),
            "cv_roc_auc_smote_std": smote_cv_scores.std(),
            "cv_roc_auc_tuned": grid_search.best_score_,
            "test_roc_auc": lr_final_test_roc,
        },
        f,
    )

print("저장 완료:", model_dir / "lr_final.pkl")


In [ ]:
# LogisticRegression Feature Importance (계수 기반)
# 모든 feature가 RobustScaler로 스케일 맞춰진 상태라 계수 크기로 상대적 영향력을 비교할 수 있음
# (양수 계수 = 사기(Class=1) 방향으로 작용, 음수 계수 = 정상(Class=0) 방향으로 작용)
importance = pd.Series(lr_final.coef_[0], index=X_train.columns).sort_values()

plt.figure(figsize=(8, 10))
colors = ["crimson" if v < 0 else "steelblue" for v in importance.values]
plt.barh(importance.index, importance.values, color=colors)
plt.title(f"LogisticRegression Feature Importance ({method_name})")
plt.xlabel("계수")
plt.tight_layout()
plt.show()

print("절댓값 기준 상위 10개:")
print(importance.reindex(importance.abs().sort_values(ascending=False).index).head(10))


In [ ]:
# L1(Lasso) 정규화 - 안 중요한 컬럼의 계수를 정확히 0으로 만들어 자동 선별
# solver="liblinear"는 L1 penalty를 지원하는 solver, 같은 skf로 C 탐색
param_grid_l1 = {"C": [0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}

grid_search_l1 = GridSearchCV(
    LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", random_state=0, max_iter=1000),
    param_grid_l1,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search_l1.fit(X_train_scaled, y_train)

print("C별 CV ROC-AUC (L1):")
for c, score in zip(param_grid_l1["C"], grid_search_l1.cv_results_["mean_test_score"]):
    print(f"  C={c}: {score:.4f}")

print(f"\n최적 C(L1): {grid_search_l1.best_params_['C']}")
print(f"최적 CV ROC-AUC(L1): {grid_search_l1.best_score_:.4f}")
print(f"CV ROC-AUC(L2, 기존 baseline): {grid_search.best_score_:.4f}")


In [ ]:
# L1 최종 재학습 -> 계수가 0이 된 컬럼 확인 -> X_test_scaled로 딱 한 번 평가
lr_l1 = LogisticRegression(
    penalty="l1", solver="liblinear",
    class_weight="balanced",
    C=grid_search_l1.best_params_["C"],
    random_state=0, max_iter=1000
)
lr_l1.fit(X_train_scaled, y_train)

l1_coef = pd.Series(lr_l1.coef_[0], index=X_train.columns)
zero_cols = l1_coef[l1_coef == 0].index.tolist()
print(f"계수가 0이 된 컬럼: {zero_cols} ({len(zero_cols)}개 / {len(l1_coef)}개)")

l1_test_roc = roc_auc_score(y_test, lr_l1.predict_proba(X_test_scaled)[:, 1])
print(f"\nL1 (C={grid_search_l1.best_params_['C']}) CV ROC-AUC:   {grid_search_l1.best_score_:.4f}")
print(f"L1 (C={grid_search_l1.best_params_['C']}) Test ROC-AUC: {l1_test_roc:.4f}")
print(f"\nL2 (기존 baseline) CV ROC-AUC:   {grid_search.best_score_:.4f}")
print(f"L2 (기존 baseline) Test ROC-AUC: {lr_final_test_roc:.4f}")


In [ ]:
# L1 vs L2 최종 비교 -> 이긴 쪽을 LogisticRegression 최종 모델로 채택, pickle 갱신
print(f"L2 (29개 컬럼 전체) Test ROC-AUC: {lr_final_test_roc:.4f}")
print(f"L1 (19개 컬럼, 10개 제거) Test ROC-AUC: {l1_test_roc:.4f}")

if l1_test_roc >= lr_final_test_roc:
    final_model, final_penalty = lr_l1, "l1"
    final_C, final_cv_score, final_test_score = grid_search_l1.best_params_["C"], grid_search_l1.best_score_, l1_test_roc
    print("\n채택: L1 (Lasso)")
else:
    final_model, final_penalty = lr_final, "l2"
    final_C, final_cv_score, final_test_score = grid_search.best_params_["C"], grid_search.best_score_, lr_final_test_roc
    print("\n채택: L2 (기존)")

with open(model_dir / "lr_final.pkl", "wb") as f:
    pickle.dump(
        {
            "model": final_model,
            "scaler": scaler,
            "penalty": final_penalty,
            "best_C": final_C,
            "cv_roc_auc": final_cv_score,
            "test_roc_auc": final_test_score,
            "zero_coef_columns": zero_cols if final_penalty == "l1" else [],
            # 참고용 - 두 방식 점수 다 같이 남김
            "l2_test_roc_auc": lr_final_test_roc,
            "l1_test_roc_auc": l1_test_roc,
        },
        f,
    )
print("저장 완료:", model_dir / "lr_final.pkl")


In [ ]:
# Average Precision (PR-AUC) - ROC-AUC와 나란히 비교
# 사기 비율이 0.17%로 극단적인 불균형 데이터라 다수 클래스(정상)에 둔감한 ROC-AUC만으로는
# "사기로 예측했을 때 실제로 사기일 확률(precision)"이 어느 정도인지 알기 어려움
# -> 소수 클래스(사기) 기준 precision-recall trade-off를 직접 요약하는 PR-AUC(average precision)를 함께 확인
from sklearn.metrics import average_precision_score

final_test_pr_auc = average_precision_score(y_test, final_model.predict_proba(X_test_scaled)[:, 1])
baseline_pr_auc = y_test.mean()  # 무작위로 찍었을 때 기대되는 PR-AUC = 실제 양성(사기) 비율

print(f"[{final_penalty.upper()} 최종 모델]")
print(f"Test ROC-AUC : {final_test_score:.4f}   (무작위 baseline = 0.50)")
print(f"Test PR-AUC  : {final_test_pr_auc:.4f}   (무작위 baseline = {baseline_pr_auc:.4f})")


#### ROC-AUC vs PR-AUC 해석

- **ROC-AUC(≈0.98)**: TPR(재현율) vs FPR(정상 데이터를 잘못 사기로 볼 확률) 관계를 요약. 정상 거래가 전체의 99.83%를 차지하는 이 데이터에서는 FPR 분모(정상 건수)가 워낙 커서, 오탐이 꽤 늘어도 FPR 자체는 잘 오르지 않음 -> ROC-AUC가 실제 운영 성능보다 낙관적으로 보일 수 있음.
- **PR-AUC(average precision)**: precision(사기로 예측한 것 중 실제 사기 비율) vs recall(실제 사기 중 잡아낸 비율) 관계를 요약. 분모가 "사기로 예측한 건수"이기 때문에 극단적 불균형에서 더 엄격하고 현실적인 지표. 무작위 분류기의 기대 PR-AUC는 실제 양성 비율(baseline_pr_auc, ≈0.0017)이므로, 모델의 PR-AUC가 이 값보다 훨씬 높다는 것 자체가 "우연이 아니라 V14 등 실제로 사기를 가르는 feature 패턴을 학습했다"는 증거.

**이 데이터/모델 실험 결과와의 연결**
- EDA에서 V14가 Class와 상관관계가 가장 큰 변수로 확인되었고(cell 11-13), 사기 케이스 기준 V14 이상치를 제거해 학습 신호를 깨끗하게 만든 것이 recall/precision 트레이드오프 개선에 직접 기여함.
- `class_weight="balanced"`/SMOTE로 불균형을 보정한 것은 ROC-AUC보다는 PR-AUC(=소수 클래스 예측 품질)를 끌어올리기 위한 조치였고, 실제로 PR-AUC가 baseline 대비 크게 높게 나오는 것으로 그 효과가 확인됨.
- L1(Lasso)이 계수 10개를 0으로 만들면서도 L2보다 Test ROC-AUC가 더 높았던 결과(cell 33)는, 이 불필요한 feature들(V1, V2, V3, V7, V9, V15, V17, V18, V19, V24)이 노이즈에 가까워 오히려 precision을 갉아먹고 있었을 가능성을 시사 -> PR-AUC로도 L1이 L2보다 우위인지 함께 보면 "변수 선택이 실제로 소수 클래스 판별력을 개선했는지"를 더 명확히 검증할 수 있음.
- 결론적으로 ROC-AUC는 "모델이 전반적으로 두 클래스를 잘 분리하는가"를, PR-AUC는 "실제 사기 탐지 운영에서 정밀도-재현율 균형이 얼마나 좋은가"를 보여주므로, 극단적 불균형 데이터에서는 PR-AUC를 주 지표로, ROC-AUC를 보조 지표로 함께 보는 것이 타당함.


### AP(average_precision) 기준으로 C 재탐색

위에서 확인했듯 지금까지 C는 전부 `roc_auc` 기준으로 골랐음(L1 최종: C=0.001). 하지만 AP(PR-AUC)를 올리는 게 목적이라면, C도 `scoring="average_precision"`으로 다시 골라야 함 -> penalty=l1, class_weight=balanced는 고정하고 C만 AP 기준으로 재탐색.

In [ ]:
# C를 average_precision 기준으로 재탐색 (penalty=l1, class_weight=balanced 고정)
# C=10, 100은 liblinear + class_weight="balanced" 조합에서 수렴이 매우 느려(fold당 수 분) 생략함
# -> 기존 L1 GridSearch(roc_auc 기준, cell a1716d86)에서 C=10/100의 CV ROC-AUC가 이미
#    C=0.001~0.1보다 낮게 나왔던 추세(0.9784)로 볼 때, AP 기준으로도 추가 이득 가능성은 낮다고 판단
from sklearn.metrics import average_precision_score

C_candidates_ap = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]
ap_cv_results = []

for c in C_candidates_ap:
    clf = LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", C=c, random_state=0, max_iter=1000)
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=skf, scoring="average_precision")
    ap_cv_results.append((c, scores.mean()))
    print(f"C={c}: CV AP={scores.mean():.4f}")

best_C_ap, best_cv_ap = max(ap_cv_results, key=lambda x: x[1])
print(f"최적 C(AP 기준): {best_C_ap}  CV AP: {best_cv_ap:.4f}")


In [ ]:
# [병합 시 수정] 원본 SH 노트북의 마지막 print에서 정의되지 않은 변수 zero_coef_columns 참조 -> zero_cols 로 교정
# AP 기준으로 고른 C로 재학습 -> X_test_scaled로 평가, 기존(roc_auc 기준 C=0.001) 모델과 비교
final_model_ap = LogisticRegression(
    penalty="l1", solver="liblinear", class_weight="balanced",
    C=best_C_ap, random_state=0, max_iter=1000
)
final_model_ap.fit(X_train_scaled, y_train)

proba_ap = final_model_ap.predict_proba(X_test_scaled)[:, 1]
test_roc_ap = roc_auc_score(y_test, proba_ap)
test_ap_ap = average_precision_score(y_test, proba_ap)

coef_ap = pd.Series(final_model_ap.coef_[0], index=X_train.columns)
zero_cols_ap = coef_ap[coef_ap == 0].index.tolist()

print(f"[AP 기준 C={best_C_ap}]")
print(f"Test ROC-AUC: {test_roc_ap:.4f}   (roc_auc 기준 C={final_C} 모델: {final_test_score:.4f})")
print(f"Test AP     : {test_ap_ap:.4f}   (roc_auc 기준 C={final_C} 모델: {final_test_pr_auc:.4f})")
print(f"0이 된 계수: {len(zero_cols_ap)}개 / {len(coef_ap)}개  (roc_auc 기준 모델: {len(zero_cols) if final_penalty=='l1' else 0}개)")


In [ ]:
# AP 기준 모델을 새 최종 모델로 채택 -> pickle 갱신 (이전 roc_auc 기준 모델 정보는 참고용으로 같이 남김)
model_dir = Path("models")

with open(model_dir / "lr_final.pkl", "wb") as f:
    pickle.dump(
        {
            "model": final_model_ap,
            "scaler": scaler,
            "penalty": "l1",
            "best_C": best_C_ap,
            "selection_metric": "average_precision",
            "cv_ap": best_cv_ap,
            "test_roc_auc": test_roc_ap,
            "test_pr_auc": test_ap_ap,
            "zero_coef_columns": zero_cols_ap,
            # 참고용 - roc_auc 기준으로 골랐던 이전 최종 모델 정보
            "prev_model_best_C": final_C,
            "prev_model_selection_metric": "roc_auc",
            "prev_model_test_roc_auc": final_test_score,
            "prev_model_test_pr_auc": final_test_pr_auc,
        },
        f,
    )
print("저장 완료:", model_dir / "lr_final.pkl", "(AP 기준 모델로 교체)")


#### 결과 요약

| | roc_auc 기준 (C=0.001) | **average_precision 기준 (C=1)** |
|---|---|---|
| Test ROC-AUC | 0.9778 | 0.9701 (소폭 하락) |
| **Test AP (PR-AUC)** | 0.6217 | **0.6606 (+0.039)** |
| 0이 된 계수 | 10개 (L1이 노이즈 컬럼 제거) | 0개 (규제가 약해져 전부 유지) |

**AP를 목적함수로 바꿔서 C를 다시 고르니 실제로 AP가 올랐음(0.6217 → 0.6606, 약 6% 개선).** 대신 ROC-AUC는 아주 조금 내려감(0.9778 → 0.9701) - 우리가 최적화하려는 지표가 AP이므로 이 트레이드오프는 감수할 만함.

C=1로 규제가 약해지면서 L1이 더 이상 계수를 0으로 만들지 않음(zero_coef_columns 0개) - 즉 AP 기준에서는 이전에 '노이즈'로 판단해 제거했던 10개 컬럼(V1, V2, V3, V7, V9, V15, V17, V18, V19, V24)도 약하게나마 사기 탐지에 기여하고 있었다는 뜻. roc_auc 기준 탐색과 average_precision 기준 탐색이 '중요한 feature가 무엇인가'에 대해서도 다른 결론을 낼 수 있음을 보여주는 사례.

**이 모델이 현재 최종 모델로 pickle(`models/lr_final.pkl`)에 저장되어 있음.**

### LGBM(트리 모델) 추가 + LR+LGBM 스태킹 검증

이혜현님의 LGBM 실험(같은 전처리 기준)에서 PR-AUC가 LR보다 훨씬 높게 나온 것을 우리 파이프라인에서도 재현하고, "LR과 LGBM을 스태킹(out-of-fold 메타모델)하면 LGBM 단독보다 더 좋아지는가"를 직접 검증한다.

In [ ]:
# LGBM 하이퍼파라미터를 average_precision 기준으로 소규모 탐색 (class_weight="balanced" 고정, 트리 모델이라 스케일링 불필요)
from lightgbm import LGBMClassifier

lgbm_configs = [
    dict(n_estimators=100, learning_rate=0.1, num_leaves=31),
    dict(n_estimators=300, learning_rate=0.05, num_leaves=31),
    dict(n_estimators=300, learning_rate=0.05, num_leaves=63),
    dict(n_estimators=300, learning_rate=0.05, num_leaves=63, reg_alpha=1.0, reg_lambda=1.0),
    dict(n_estimators=500, learning_rate=0.03, num_leaves=63),
]

lgbm_cv_results = []
for cfg in lgbm_configs:
    clf = LGBMClassifier(random_state=0, n_jobs=-1, verbose=-1, class_weight="balanced", **cfg)
    scores = cross_val_score(clf, X_train, y_train, cv=skf, scoring="average_precision")
    lgbm_cv_results.append((cfg, scores.mean()))
    print(f"{cfg} -> CV AP={scores.mean():.4f}")

best_lgbm_cfg, best_lgbm_cv_ap = max(lgbm_cv_results, key=lambda r: r[1])
print("")
print(f"최적 LGBM 설정: {best_lgbm_cfg}  CV AP: {best_lgbm_cv_ap:.4f}")


In [ ]:
# 탐색된 최적 설정으로 LGBM 최종 학습 -> X_test로 평가 -> pickle 저장
lgbm_final = LGBMClassifier(
    random_state=0, n_jobs=-1, verbose=-1, class_weight="balanced", **best_lgbm_cfg
)
lgbm_final.fit(X_train, y_train)

proba_lgbm_test = lgbm_final.predict_proba(X_test)[:, 1]
lgbm_test_roc = roc_auc_score(y_test, proba_lgbm_test)
lgbm_test_ap = average_precision_score(y_test, proba_lgbm_test)

print(f"LGBM Test ROC-AUC: {lgbm_test_roc:.4f}")
print(f"LGBM Test AP     : {lgbm_test_ap:.4f}")
print(f"(참고) LR(AP 기준 최종) Test AP: {test_ap_ap:.4f}")

with open(model_dir / "lgbm_final.pkl", "wb") as f:
    pickle.dump(
        {
            "model": lgbm_final,
            "params": best_lgbm_cfg,
            "cv_ap": best_lgbm_cv_ap,
            "test_roc_auc": lgbm_test_roc,
            "test_pr_auc": lgbm_test_ap,
        },
        f,
    )
print("저장 완료:", model_dir / "lgbm_final.pkl")


In [ ]:
# 스태킹을 위한 out-of-fold(OOF) 예측 생성 (LR, LGBM 둘 다 -> 학습 데이터 누수 없이 메타모델용 feature 생성)
from sklearn.model_selection import cross_val_predict

oof_lr = cross_val_predict(
    LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", C=best_C_ap, random_state=0, max_iter=1000),
    X_train_scaled, y_train, cv=skf, method="predict_proba"
)[:, 1]

oof_lgbm = cross_val_predict(
    LGBMClassifier(random_state=0, n_jobs=-1, verbose=-1, class_weight="balanced", **best_lgbm_cfg),
    X_train, y_train, cv=skf, method="predict_proba"
)[:, 1]

meta_X_train = pd.DataFrame({"lr": oof_lr, "lgbm": oof_lgbm}, index=X_train.index)
meta_X_test = pd.DataFrame({"lr": proba_ap, "lgbm": proba_lgbm_test}, index=X_test.index)

print("OOF 예측 생성 완료:", meta_X_train.shape)


In [ ]:
# 메타모델(LogisticRegression) 학습 -> class_weight 옵션 비교 후 최종 평가
for cw in [None, "balanced"]:
    meta_clf = LogisticRegression(random_state=0, class_weight=cw)
    scores = cross_val_score(meta_clf, meta_X_train, y_train, cv=skf, scoring="average_precision")
    print(f"meta class_weight={cw} -> CV AP={scores.mean():.4f}")

meta_model = LogisticRegression(random_state=0)  # 두 옵션 CV AP 차이가 거의 없어 기본값 채택
meta_model.fit(meta_X_train, y_train)

proba_stack_test = meta_model.predict_proba(meta_X_test)[:, 1]
stack_test_roc = roc_auc_score(y_test, proba_stack_test)
stack_test_ap = average_precision_score(y_test, proba_stack_test)

print("")
print(f"메타모델 계수: LR={meta_model.coef_[0][0]:.4f}, LGBM={meta_model.coef_[0][1]:.4f}")
print(f"Stacking Test ROC-AUC: {stack_test_roc:.4f}")
print(f"Stacking Test AP     : {stack_test_ap:.4f}")


In [ ]:
# 최종 비교: LR vs LGBM vs Stacking (+참고: 가중평균 블렌딩) -> stacking_final.pkl로 저장
print(f"LR       : Test ROC-AUC={test_roc_ap:.4f}  Test AP={test_ap_ap:.4f}")
print(f"LGBM     : Test ROC-AUC={lgbm_test_roc:.4f}  Test AP={lgbm_test_ap:.4f}")
print(f"Stacking : Test ROC-AUC={stack_test_roc:.4f}  Test AP={stack_test_ap:.4f}")
print("")

for w in [0.0, 0.1, 0.2, 0.3, 0.5]:
    blend = w * proba_ap + (1 - w) * proba_lgbm_test
    print(f"가중평균(LR비중={w}) Test AP: {average_precision_score(y_test, blend):.4f}")

with open(model_dir / "stacking_final.pkl", "wb") as f:
    pickle.dump(
        {
            "lr_model": final_model_ap,
            "lgbm_model": lgbm_final,
            "meta_model": meta_model,
            "scaler": scaler,
            "lr_test_roc_auc": test_roc_ap,
            "lr_test_pr_auc": test_ap_ap,
            "lgbm_test_roc_auc": lgbm_test_roc,
            "lgbm_test_pr_auc": lgbm_test_ap,
            "stack_test_roc_auc": stack_test_roc,
            "stack_test_pr_auc": stack_test_ap,
        },
        f,
    )
print("")
print("저장 완료:", model_dir / "stacking_final.pkl")


#### 결론: 스태킹은 LGBM 단독을 못 이김

| 모델 | Test ROC-AUC | Test AP |
|---|---|---|
| LR (AP 기준 최종) | 0.9701 | 0.6606 |
| **LGBM (단독)** | 0.9644 | **0.8255** |
| LR+LGBM 스태킹 | 0.9703 | 0.8145 |
| 가중평균 (LR 10%) | - | 0.8154 |

**LGBM 단독(0.8255)이 스태킹(0.8145)이나 어떤 가중평균 조합보다도 AP가 높습니다.** LR을 조금이라도 섞을수록(가중치를 늘릴수록) AP가 계속 떨어지는 걸 보면(0.8255 → 0.8154 → 0.8153 → 0.8146 → 0.8144), LR의 예측이 LGBM에 도움이 되기는커녕 노이즈로 작용하고 있다는 뜻.

이전에 얘기했던 "두 모델 실력 격차가 크면(LR 0.66 vs LGBM 0.83) 다양성보다 약한 모델이 발목 잡는 효과가 더 크다"는 우려가 실제로 확인된 사례. 메타모델이 LR 계수(5.72)를 LGBM 계수(6.52)와 비슷한 크기로 살려뒀는데도(LR을 완전히 무시하지 않았는데도) LGBM 단독보다 낮게 나온 걸 보면, 이 데이터·이 두 모델 조합에서는 스태킹의 이득이 없다고 결론지을 수 있음.

**현재 이 데이터에서 PR-AUC 기준 최고 성능 모델은 `models/lgbm_final.pkl`의 LGBM 단독 모델(Test AP=0.8255)이며, 세 모델(LR/LGBM/스태킹)은 각각 `models/lr_final.pkl`, `models/lgbm_final.pkl`, `models/stacking_final.pkl`에 전부 저장되어 있어 다시 불러올 수 있음.**

## [권용우] Logistic Regression

In [ ]:
# 이 섹션 작업용 데이터 — 공통 원본에서 복사 (한 사람의 전처리가 다음 사람에게 영향 주지 않도록 격리)
train_df = _raw.copy()

In [ ]:
# 데이터 로드 및 기본 정보 확인
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

plt.rcParams['font.family'] = 'Malgun Gothic'  # 그래프에 한글이 깨지지 않도록 폰트 설정 (Windows 기본 맑은 고딕)
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트 사용 시 마이너스(-) 기호 깨짐 방지

train_df.head(3)

In [ ]:
# Time 컬럼 제거
# Time은 거래 발생 후 경과 초(0~172792)라 사기 여부와 인과관계가 약하고,
# 스케일링되지 않은 큰 범위 값이라 V1~V28(대부분 -1~1대) 옆에 같이 들어가면 로지스틱 회귀 최적화(수렴)에 방해가 됨.
# 실제 벤치마크(4-2 데이터 기준): Time 유지 시 AUC 0.9553(수렴 경고 발생) -> Time 제거 시 AUC 0.9744로 개선 확인.
# 이후 모든 실험(1~7단계, 튜닝, SMOTE 포함)은 Time이 제거된 데이터로 진행.
train_df = train_df.drop('Time', axis=1)
train_df.head(3)

In [ ]:
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score


def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
train_df.info()

In [ ]:
train_df.describe().T

In [ ]:
# 공통 함수 정의: 이상치 확인/제거, 중복행 제거, 스케일링, 로지스틱 회귀 학습+평가
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


def get_outlier(df, column, weight=1.5):
    fraud = df[df['Class'] == 1][column]  # 사기(Class=1) 건의 해당 컬럼 값만 추출
    q25 = np.percentile(fraud.values, 25)  # 1사분위수
    q75 = np.percentile(fraud.values, 75)  # 3사분위수
    iqr = (q75 - q25) * weight  # IQR에 가중치를 곱한 값
    lowest = q25 - iqr  # 하한
    highest = q75 + iqr  # 상한
    return fraud[(fraud < lowest) | (fraud > highest)].index  # 하한보다 작거나 상한보다 큰 값의 인덱스 반환


def remove_outliers(df):
    df_out = df.copy()  # 원본 보존을 위해 복사
    idx = get_outlier(df_out, 'V14', weight=1.5)  # V14 기준 이상치 인덱스 추출
    return df_out.drop(idx, axis=0)  # 이상치 행 제거


def remove_dup(df):
    return df.drop_duplicates().reset_index(drop=True)  # 중복행 제거 후 인덱스 재정렬


def scale_amount(df):
    df_s = df.copy()  # 원본 보존을 위해 복사
    df_s['Amount_Scaled'] = RobustScaler().fit_transform(df_s[['Amount']])  # Amount를 RobustScaler로 스케일링
    return df_s.drop('Amount', axis=1)  # 스케일링 전 원본 Amount 컬럼 삭제


def run_lr(df, name):
    X = df.drop('Class', axis=1)  # 피처
    y = df['Class']  # 레이블(사기 여부)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0, stratify=y
    )  # 학습/테스트 분리 (모든 실험 동일 옵션 사용)
    model = LogisticRegression(max_iter=1000)  # 로지스틱 회귀 모델 생성
    model.fit(X_train, y_train)  # 학습
    pred = model.predict(X_test)  # 예측 클래스(0/1)
    proba = model.predict_proba(X_test)[:, 1]  # 사기(Class=1) 예측 확률
    print(f'--- {name} ---')
    get_clf_eval(y_test, pred, proba)  # 성능 평가 출력
    return {'model': model, 'pred': pred, 'proba': proba}

### 1. 원본 데이터 (전처리 없음)

In [ ]:
# 1-1. 원본 그대로 (스케일링 X)
res_raw = run_lr(train_df, '원본(스케일링 X)')  # 이상치/중복행/Amount 모두 원본 그대로 사용

In [ ]:
# 1-2. 원본 + 스케일링만 적용
df_raw_scaled = scale_amount(train_df)  # 이상치/중복행 제거 없이 Amount 스케일링만 적용
res_raw_scaled = run_lr(df_raw_scaled, '원본+스케일링')

### 2. 이상치만 제거 (중복행은 그대로)

In [ ]:
# V14 이상치 시각적 확인 (사기 거래 기준)
fraud_v14 = train_df[train_df['Class'] == 1]['V14']  # 사기 거래(Class=1)의 V14 값만 추출
q25, q75 = fraud_v14.quantile(0.25), fraud_v14.quantile(0.75)  # 1사분위수, 3사분위수
lower_bound = q25 - 1.5 * (q75 - q25)  # 이상치 판정 하한선 (get_outlier와 동일한 기준: IQR*1.5)

plt.figure(figsize=(5, 4))
plt.boxplot(fraud_v14)  # 박스=IQR(25~75%), 가운데 선=중앙값, 수염 밖 점(o)=이상치
plt.axhline(lower_bound, color='red', linestyle='--', linewidth=1, label=f'이상치 하한선 ({lower_bound:.2f})')  # 하한선을 그래프에 표시
plt.title('V14 boxplot (Class=1)')
plt.legend()
plt.show()

print(fraud_v14.describe())  # 사기 거래 V14의 기초통계량 확인

outlier_idx = get_outlier(train_df, 'V14', weight=1.5)  # IQR 기준 이상치 인덱스 추출
print('이상치 인덱스:', outlier_idx.tolist())
print('이상치 V14 값:', train_df.loc[outlier_idx, 'V14'].tolist())  # 하한선보다 얼마나 벗어났는지 실제 값으로 확인

In [ ]:
# 2-1. 이상치 제거 (스케일링 X)
df_out = remove_outliers(train_df)  # 이상치만 제거, 중복행/스케일링은 그대로
res_out = run_lr(df_out, '이상치제거(스케일링 X)')

In [ ]:
# 2-2. 이상치 제거 + 스케일링
df_out_scaled = scale_amount(df_out)  # 이상치 제거된 데이터에 스케일링 추가 적용
res_out_scaled = run_lr(df_out_scaled, '이상치제거+스케일링')

### 3. 중복행만 제거 (이상치는 그대로)

In [ ]:
# 중복행 확인
print('전체 행 수:', len(train_df))
print('중복행 수:', train_df.duplicated().sum())  # 완전히 동일한 행이 몇 개인지 확인

In [ ]:
# 3-1. 중복행 제거 (스케일링 X)
df_dup = remove_dup(train_df)  # 중복행만 제거, 이상치/스케일링은 그대로
res_dup = run_lr(df_dup, '중복제거(스케일링 X)')

In [ ]:
# 3-2. 중복행 제거 + 스케일링
df_dup_scaled = scale_amount(df_dup)  # 중복행 제거된 데이터에 스케일링 추가 적용
res_dup_scaled = run_lr(df_dup_scaled, '중복제거+스케일링')

### 4. 이상치 + 중복행 모두 제거
※ 이상치·중복행은 위에서 이미 시각적으로 확인했으므로 바로 제거 후 진행.

In [ ]:
# 4-1. 이상치+중복행 모두 제거 (스케일링 X)
df_both = remove_dup(remove_outliers(train_df))  # 이상치 제거 후 중복행까지 제거
res_both = run_lr(df_both, '이상치+중복제거(스케일링 X)')

In [ ]:
# 4-2. 이상치+중복행 모두 제거 + 스케일링
df_both_scaled = scale_amount(df_both)  # 이상치+중복행 제거된 데이터에 스케일링 추가 적용
res_both_scaled = run_lr(df_both_scaled, '이상치+중복제거+스케일링')

### 결과 저장 (피클)

In [ ]:
# 8개 실험 결과를 하나의 딕셔너리로 묶어 피클로 저장
import pickle

results = {
    'raw': res_raw,                    # 1-1. 원본
    'raw_scaled': res_raw_scaled,      # 1-2. 원본+스케일링
    'outlier': res_out,                # 2-1. 이상치제거
    'outlier_scaled': res_out_scaled,  # 2-2. 이상치제거+스케일링
    'dup': res_dup,                    # 3-1. 중복제거
    'dup_scaled': res_dup_scaled,      # 3-2. 중복제거+스케일링
    'both': res_both,                  # 4-1. 이상치+중복제거
    'both_scaled': res_both_scaled,    # 4-2. 이상치+중복제거+스케일링
}

with open('lr_clf_result.pkl', 'wb') as f:  # 바이너리 쓰기 모드로 파일 오픈
    pickle.dump(results, f)  # results 딕셔너리를 피클로 직렬화하여 저장

print('저장 완료: lr_clf_result.pkl')

### 5. 4-2 데이터로 하이퍼파라미터 튜닝 (AUC·F1 최적화)
이상치+중복행 제거 + 스케일링까지 적용한 `df_both_scaled`를 그대로 사용한다.
`GridSearchCV`로 C·penalty·class_weight를 탐색해 AUC를 최대화하고,
그 다음 예측 확률의 판정 임계값(threshold)을 별도로 튜닝해 F1까지 끌어올린다.

In [ ]:
# 5-1. GridSearchCV로 AUC 기준 최적 하이퍼파라미터 탐색
from sklearn.model_selection import GridSearchCV

X_tune = df_both_scaled.drop('Class', axis=1)  # 4-2 데이터의 피처
y_tune = df_both_scaled['Class']  # 4-2 데이터의 레이블(사기 여부)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tune, y_tune, test_size=0.2, random_state=0, stratify=y_tune
)  # 기존 실험들과 동일한 분리 옵션

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],        # 규제 강도 (작을수록 규제가 강함)
    'penalty': ['l1', 'l2'],              # 규제 방식
    'class_weight': [None, 'balanced'],   # 클래스 불균형 보정 여부 (사기 비율 0.17%라 효과가 클 수 있음)
}

grid = GridSearchCV(
    LogisticRegression(solver='liblinear', max_iter=1000),  # l1/l2 규제를 모두 지원하는 solver
    param_grid,
    scoring='roc_auc',  # AUC를 기준으로 최적 조합 탐색
    cv=5,                # 5-fold 교차검증
    n_jobs=-1,            # 가용 CPU 코어를 모두 사용해 병렬 탐색
)
grid.fit(X_tr, y_tr)  # 학습 데이터에서만 탐색 (테스트셋은 아직 손대지 않음)

print('베스트 파라미터:', grid.best_params_)
print('베스트 CV AUC:', round(grid.best_score_, 4))

In [ ]:
# 5-2. 튜닝된 모델을 테스트셋에 적용 (기본 임계값 0.5)
best_model = grid.best_estimator_  # GridSearchCV가 전체 학습 데이터로 재학습해둔 최적 모델
proba_te = best_model.predict_proba(X_te)[:, 1]  # 테스트셋 사기 예측 확률
pred_te = best_model.predict(X_te)  # 기본 임계값(0.5) 기준 예측 클래스

print('--- 튜닝 모델 (threshold=0.5) ---')
get_clf_eval(y_te, pred_te, proba_te)

In [ ]:
# 5-3. F1을 최대화하는 임계값 탐색 (학습 데이터의 교차검증 예측확률만 사용, 테스트셋은 건드리지 않음)
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

oof_proba = cross_val_predict(
    best_model, X_tr, y_tr, cv=5, method='predict_proba', n_jobs=-1
)[:, 1]  # 학습 데이터 각 행에 대한 out-of-fold 예측 확률 (자기 자신을 학습에 쓰지 않은 예측)

precisions, recalls, thresholds = precision_recall_curve(y_tr, oof_proba)  # 임계값별 정밀도/재현율
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)  # 임계값별 F1 계산
best_idx = f1s[:-1].argmax()  # precision_recall_curve는 thresholds가 precisions/recalls보다 1개 적음
best_threshold = thresholds[best_idx]  # F1이 가장 높은 임계값

print(f'F1 최적 임계값: {best_threshold:.4f} (교차검증 F1={f1s[best_idx]:.4f})')

In [ ]:
# 5-4. 튜닝된 임계값을 테스트셋에 적용해 최종 평가
pred_tuned = (proba_te >= best_threshold).astype(int)  # 확률이 튜닝된 임계값 이상이면 사기(1)로 판정

print('--- 튜닝 모델 (threshold 최적화) ---')
get_clf_eval(y_te, pred_tuned, proba_te)

res_tuned = {
    'model': best_model,
    'pred': pred_tuned,
    'proba': proba_te,
    'threshold': best_threshold,
    'params': grid.best_params_,
}

In [ ]:
# 튜닝 결과까지 포함해서 피클 다시 저장
results['tuned_4_2'] = res_tuned  # 4-2 데이터 + 하이퍼파라미터/임계값 튜닝 결과 추가

with open('lr_clf_result.pkl', 'wb') as f:
    pickle.dump(results, f)

print('저장 완료: lr_clf_result.pkl (튜닝 결과 포함)')

### 6. Hyperopt(TPE)로 하이퍼파라미터 재탐색
GridSearchCV는 격자 위 모든 조합을 다 시도하는 전수 탐색이다. Hyperopt는 지금까지의 시도 결과를 바탕으로
다음에 시도할 조합을 베이지안 방식(TPE)으로 골라, 같은 데이터·같은 탐색 범위에서 GridSearchCV와
결과를 비교한다.

In [ ]:
# 6-1. Hyperopt(TPE)로 C·penalty·class_weight 재탐색 (GridSearchCV와 동일한 데이터·범위로 비교)
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.model_selection import cross_val_score

space = {
    'C': hp.loguniform('C', np.log(0.01), np.log(100)),           # GridSearchCV의 0.01~100 범위와 동일한 로그스케일 연속값
    'penalty': hp.choice('penalty', ['l1', 'l2']),                  # 규제 방식
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 불균형 보정 여부
}

def objective(params):
    clf = LogisticRegression(solver='liblinear', max_iter=1000, **params)  # GridSearchCV와 동일 설정
    auc = cross_val_score(clf, X_tr, y_tr, cv=3, scoring='roc_auc', n_jobs=-1).mean()  # 3-fold CV 평균 AUC
    return {'loss': -auc, 'status': STATUS_OK}  # hyperopt는 손실을 최소화하므로 AUC에 음수를 취함

trials = Trials()
best_raw = fmin(
    objective, space, algo=tpe.suggest, max_evals=15, trials=trials,
    rstate=np.random.default_rng(0),  # 재현 가능한 결과를 위한 시드
)
best_params_hopt = space_eval(space, best_raw)  # 인덱스로 나온 결과를 실제 파라미터 값으로 변환

print('Hyperopt 베스트 파라미터:', best_params_hopt)
print('Hyperopt 베스트 CV AUC:', round(-min(t['result']['loss'] for t in trials.trials), 4))

In [ ]:
# 6-2. Hyperopt가 찾은 최적 파라미터로 재학습 후 테스트셋 평가 (threshold=0.5)
best_model_hopt = LogisticRegression(solver='liblinear', max_iter=1000, **best_params_hopt)
best_model_hopt.fit(X_tr, y_tr)

proba_te_hopt = best_model_hopt.predict_proba(X_te)[:, 1]  # 테스트셋 사기 예측 확률
pred_te_hopt = best_model_hopt.predict(X_te)  # 기본 임계값(0.5) 기준 예측 클래스

print('--- Hyperopt 튜닝 모델 (threshold=0.5) ---')
get_clf_eval(y_te, pred_te_hopt, proba_te_hopt)

In [ ]:
# 6-3. Hyperopt 모델도 동일한 방식으로 F1 최적 임계값 탐색 후 최종 평가
oof_proba_hopt = cross_val_predict(
    best_model_hopt, X_tr, y_tr, cv=5, method='predict_proba', n_jobs=-1
)[:, 1]  # 학습 데이터 out-of-fold 예측 확률 (테스트셋은 사용하지 않음)

precisions_h, recalls_h, thresholds_h = precision_recall_curve(y_tr, oof_proba_hopt)
f1s_h = 2 * precisions_h * recalls_h / (precisions_h + recalls_h + 1e-12)
best_idx_h = f1s_h[:-1].argmax()
best_threshold_hopt = thresholds_h[best_idx_h]  # F1이 가장 높은 임계값

print(f'Hyperopt 모델 F1 최적 임계값: {best_threshold_hopt:.4f} (교차검증 F1={f1s_h[best_idx_h]:.4f})')

pred_tuned_hopt = (proba_te_hopt >= best_threshold_hopt).astype(int)  # 튜닝된 임계값을 테스트셋에 적용
print('--- Hyperopt 튜닝 모델 (threshold 최적화) ---')
get_clf_eval(y_te, pred_tuned_hopt, proba_te_hopt)

res_tuned_hopt = {
    'model': best_model_hopt,
    'pred': pred_tuned_hopt,
    'proba': proba_te_hopt,
    'threshold': best_threshold_hopt,
    'params': best_params_hopt,
}

In [ ]:
# 6-4. GridSearchCV vs Hyperopt 비교 + 피클 저장
print('GridSearchCV :', grid.best_params_, '| test AUC', round(roc_auc_score(y_te, proba_te), 4), '| tuned F1', round(f1_score(y_te, pred_tuned), 4))
print('Hyperopt     :', best_params_hopt, '| test AUC', round(roc_auc_score(y_te, proba_te_hopt), 4), '| tuned F1', round(f1_score(y_te, pred_tuned_hopt), 4))

results['tuned_4_2_hyperopt'] = res_tuned_hopt  # Hyperopt로 튜닝한 결과 추가

with open('lr_clf_result.pkl', 'wb') as f:
    pickle.dump(results, f)

print('저장 완료: lr_clf_result.pkl (Hyperopt 결과 포함)')

### 7. Hyperopt 베스트 파라미터 + SMOTE
Hyperopt가 찾은 `best_params_hopt`(C=0.0249, class_weight='balanced', penalty='l1')는 그대로 두고,
학습 데이터에만 SMOTE로 소수 클래스(사기)를 오버샘플링해서 같은 4-2 데이터에서 결과가 어떻게 달라지는지 확인한다.
테스트셋은 원본 그대로 유지해 왜곡 없이 평가한다.

In [ ]:
# 7-1. Hyperopt 베스트 파라미터로 SMOTE 적용 데이터 학습 (SMOTE는 학습 데이터에만 적용)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)  # 학습 데이터만 오버샘플링, 테스트셋은 원본 그대로 유지

print('SMOTE 적용 전 학습 데이터 클래스 분포:', y_tr.value_counts().to_dict())
print('SMOTE 적용 후 학습 데이터 클래스 분포:', y_tr_sm.value_counts().to_dict())

model_smote = LogisticRegression(solver='liblinear', max_iter=1000, **best_params_hopt)  # Hyperopt 베스트 파라미터 그대로 사용
model_smote.fit(X_tr_sm, y_tr_sm)  # SMOTE로 균형 맞춘 데이터로 학습

proba_te_smote = model_smote.predict_proba(X_te)[:, 1]  # 테스트셋(원본, 미변형)에 대한 예측 확률
pred_te_smote = model_smote.predict(X_te)  # 기본 임계값(0.5) 기준 예측 클래스

print('--- Hyperopt 베스트 파라미터 + SMOTE (threshold=0.5) ---')
get_clf_eval(y_te, pred_te_smote, proba_te_smote)

In [ ]:
# 7-2. F1 최적 임계값 탐색 (SMOTE를 fold 안에서만 적용해 데이터 누수 방지) 후 테스트셋 최종 평가
from imblearn.pipeline import Pipeline as ImbPipeline

smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=0)),
    ('clf', LogisticRegression(solver='liblinear', max_iter=1000, **best_params_hopt)),
])  # 파이프라인으로 묶어야 교차검증 각 fold 안에서만 SMOTE가 적용되어 검증 폴드로의 누수를 막을 수 있음

oof_proba_smote = cross_val_predict(
    smote_pipe, X_tr, y_tr, cv=5, method='predict_proba', n_jobs=-1
)[:, 1]  # 원본(오버샘플링 전) 학습 데이터 기준 out-of-fold 예측 확률

precisions_s, recalls_s, thresholds_s = precision_recall_curve(y_tr, oof_proba_smote)
f1s_s = 2 * precisions_s * recalls_s / (precisions_s + recalls_s + 1e-12)
best_idx_s = f1s_s[:-1].argmax()
best_threshold_smote = thresholds_s[best_idx_s]  # F1이 가장 높은 임계값

print(f'SMOTE 모델 F1 최적 임계값: {best_threshold_smote:.4f} (교차검증 F1={f1s_s[best_idx_s]:.4f})')

pred_tuned_smote = (proba_te_smote >= best_threshold_smote).astype(int)  # 튜닝된 임계값을 테스트셋에 적용
print('--- Hyperopt 베스트 파라미터 + SMOTE (threshold 최적화) ---')
get_clf_eval(y_te, pred_tuned_smote, proba_te_smote)

res_smote = {
    'model': model_smote,
    'pred': pred_tuned_smote,
    'proba': proba_te_smote,
    'threshold': best_threshold_smote,
    'params': best_params_hopt,
}

In [ ]:
# 7-3. Hyperopt(SMOTE 없음) vs Hyperopt+SMOTE 비교 + 피클 저장
print('Hyperopt         :', best_params_hopt, '| test AUC', round(roc_auc_score(y_te, proba_te_hopt), 4), '| tuned F1', round(f1_score(y_te, pred_tuned_hopt), 4))
print('Hyperopt + SMOTE :', best_params_hopt, '| test AUC', round(roc_auc_score(y_te, proba_te_smote), 4), '| tuned F1', round(f1_score(y_te, pred_tuned_smote), 4))

results['tuned_4_2_hyperopt_smote'] = res_smote  # Hyperopt 베스트 파라미터 + SMOTE 결과 추가

with open('lr_clf_result.pkl', 'wb') as f:
    pickle.dump(results, f)

print('저장 완료: lr_clf_result.pkl (SMOTE 결과 포함)')